# V16: Multi-GNN — Full Scale-Free Graph (Patient + Gene layers) + Graph XAI

**What's new in V16:**
- **Dual Scale-Free Layers**: Gene-Gene co-mutation BA edges added alongside the existing Patient-Patient BA layer. Both sides of the bipartite graph now exhibit hub structure.
- **Graph Visualisation**: Interactive subgraph view showing all three edge types (Gene↔Patient bipartite, Patient-Patient BA, Gene-Gene BA) with degree-proportional node sizes.
- **Model Updates**: HeteroGATv2 and MOGAT gain a `gg_conv` GCN layer that propagates gene embeddings over the Gene-Gene co-mutation graph before message-passing to patients.
- **Graph-Specific XAI** replaces global permutation importance: edge occlusion, integrated gradients on clinical features, population-level attribution, and a GNNExplainer-style case study with cross-dataset gene overlap.
- **Comprehensive Attention Suite**: 7 attention views for HeteroGATv2 and MOGAT including per-head breakdown, entropy, bidirectional (g2p + p2g), MOGAT fusion gate, and cross-pipeline stability.
- **Bug fixes**: V11→V15 file prefix corrections, ROS re-run guard.


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random, copy, warnings

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score, accuracy_score)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from torch_geometric.data import HeteroData
from torch_geometric.nn import (GATv2Conv, GCNConv, Linear, HGTConv,
                                 RGCNConv, HypergraphConv, ChebConv)

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import Condition
from imblearn.over_sampling import SMOTENC
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import os

def set_seed(seed=42):
    """Full determinism:
    - Python / NumPy / PyTorch RNGs
    - cuDNN deterministic algorithms (no non-deterministic scatter ops)
    - PYTHONHASHSEED for dict/set ordering
    NOTE: cudnn.deterministic=True adds ~10-20% training overhead.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # no non-det cuDNN kernels
    torch.backends.cudnn.benchmark     = False  # disable auto-tuner (picks diff algo each run)

set_seed(42)

N_TRIALS   = 30    # Optuna TPE trials per model×pipeline
N_FOLDS    = 5     # stratified CV folds
MAX_EPOCHS = 200
PATIENCE   = 20

/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: libnvToolsExt.so.1: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/nafim/anaconda3/envs/thesis/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


Using device: cuda
GPU: NVIDIA GeForce RTX 3050


## 2. Load Data

In [2]:
tcga_df = pd.read_csv('../../dataset/TCGA_InfoWithGrade.csv')
cgga_df = pd.read_csv('../../dataset/weseq_processed_with_id_and_race_V2.csv')

if 'CGGA_ID' in cgga_df.columns:
    cgga_df = cgga_df.drop(columns='CGGA_ID')

categorical_columns = ['Grade', 'Gender', 'Race']
gene_columns = ['IDH1','TP53','ATRX','PTEN','EGFR','CIC','MUC16','PIK3CA',
                'NF1','PIK3R1','FUBP1','RB1','NOTCH1','BCOR','CSMD3',
                'SMARCA4','GRIN2A','IDH2','FAT4','PDGFRA']
NUM_GENES = len(gene_columns)

print("TCGA:", tcga_df.shape, "  Grade dist:", dict(tcga_df['Grade'].value_counts()))
print("CGGA:", cgga_df.shape, "  Grade dist:", dict(cgga_df['Grade'].value_counts()))

TCGA: (839, 24)   Grade dist: {0: 487, 1: 352}
CGGA: (286, 24)   Grade dist: {0: 184, 1: 102}


## 3. Splits — 20% held-out test; remaining 80% used for HPO + 5-Fold CV

In [3]:
# Hold out 20% as final test set (never seen during HPO or CV)
train_val_df, test_df = train_test_split(
    tcga_df, test_size=0.2, stratify=tcga_df['Grade'], random_state=42)

# Further split train_val for Optuna HPO warmup (val graph used only in HPO phase)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2, stratify=train_val_df['Grade'], random_state=42)

print(f"Train(HPO)={len(train_df)}  Val(HPO)={len(val_df)}  "
      f"TrainVal(CV)={len(train_val_df)}  Test={len(test_df)}")
print("Train Grade dist:", dict(train_df['Grade'].value_counts()))
print("Test  Grade dist:", dict(test_df['Grade'].value_counts()))

Train(HPO)=536  Val(HPO)=135  TrainVal(CV)=671  Test=168
Train Grade dist: {0: 311, 1: 225}
Test  Grade dist: {0: 98, 1: 70}


## 4. Graph Construction — Bipartite + Dual Scale-Free Layers

In [4]:
_PP_CACHE: dict = {}


def construct_bipartite_heterograph(df: pd.DataFrame) -> HeteroData:
    graph = HeteroData()
    # ── Patient feature encoding (correct) ────────────────────────────────
    # Gender      : binary 0/1          -> 1 feature (already correct)
    # Race        : nominal {0,1,2,3}   -> one-hot 4 features
    #               (raw int encoding is wrong: implies White<Asian<Black<Other)
    # Age_at_diag : continuous float    -> StandardScaler normalize -> 1 feature
    # Total: Patient.x = [Gender | Race_0 Race_1 Race_2 Race_3 | Age_norm]  shape [N, 6]
    scaler   = StandardScaler()
    age_norm = scaler.fit_transform(df[['Age_at_diagnosis']].values.astype(float))  # [N,1]
    gender   = df[['Gender']].values.astype(float)                                  # [N,1]
    race_ohe = np.zeros((len(df), 4), dtype=float)
    for i, rv in enumerate(df['Race'].values.astype(int)):
        if 0 <= rv < 4:
            race_ohe[i, rv] = 1.0                                                   # [N,4]
    pat_feats = np.hstack([gender, race_ohe, age_norm])                             # [N,6]
    graph['Patient'].x = torch.tensor(pat_feats, dtype=torch.float)
    graph['Patient'].y = torch.tensor(df['Grade'].values, dtype=torch.long)
    graph['Gene'].x    = torch.eye(NUM_GENES, dtype=torch.float)

    src_genes, dst_patients = [], []
    for p_idx, (_, row) in enumerate(df.iterrows()):
        for g_idx, gene in enumerate(gene_columns):
            if int(row[gene]) == 1:
                src_genes.append(g_idx); dst_patients.append(p_idx)

    graph['Gene',    'mutates',    'Patient'].edge_index = torch.tensor([src_genes,    dst_patients], dtype=torch.long)
    graph['Patient', 'mutated_by', 'Gene'   ].edge_index = torch.tensor([dst_patients, src_genes   ], dtype=torch.long)
    return graph


def get_pp_edges(graph: HeteroData, max_pts: int = 30):
    key = id(graph)
    if key in _PP_CACHE:
        return _PP_CACHE[key]
    ei = graph[('Gene','mutates','Patient')].edge_index.cpu()
    gene_ids, pat_ids = ei[0], ei[1]
    n_p = graph['Patient'].x.shape[0]
    all_src, all_dst, all_types = [], [], []
    torch.manual_seed(42)
    for g in range(NUM_GENES):
        mask = (gene_ids == g)
        pts = pat_ids[mask].unique(); n = len(pts)
        if n < 2: continue
        if n > max_pts:
            pts = pts[torch.randperm(n)[:max_pts]]; n = max_pts
        src = pts.repeat_interleave(n); dst = pts.repeat(n)
        valid = (src != dst)
        all_src.append(src[valid]); all_dst.append(dst[valid])
        all_types.append(torch.full((valid.sum(),), g, dtype=torch.long))
    if all_src:
        pp_ei = torch.stack([torch.cat(all_src), torch.cat(all_dst)])
        pp_et = torch.cat(all_types)
    else:
        idx = torch.arange(n_p)
        pp_ei = torch.stack([idx, idx]); pp_et = torch.zeros(n_p, dtype=torch.long)
    _PP_CACHE[key] = (pp_ei, pp_et)
    return pp_ei, pp_et


def to_dev(graph): return graph.to(device)


def clear_pp_cache():
    """Call between CV folds to free stale graph-id entries."""
    _PP_CACHE.clear()

## 4a. Scale-Free Analysis — Degree Distribution Proof

In [ ]:
from scipy import stats

def get_degree_sequences(graph):
    ei       = graph[("Gene","mutates","Patient")].edge_index.cpu()
    gene_ids = ei[0].numpy(); pat_ids = ei[1].numpy()
    pat_deg  = np.bincount(pat_ids,  minlength=graph["Patient"].x.shape[0])
    gene_deg = np.bincount(gene_ids, minlength=graph["Gene"].x.shape[0])
    return pat_deg, gene_deg


def _hill_mle(degrees, k_min=None):
    """Hill (1975) MLE for power-law exponent — unbiased for any N."""
    d = degrees[degrees > 0].astype(float)
    if k_min is None: k_min = d.min()
    tail = d[d >= k_min]
    if len(tail) < 2: return 0.0
    return 1.0 + len(tail) / np.sum(np.log(tail / (k_min - 0.5)))


def fit_powerlaw(degrees, label="Degrees"):
    """
    Two-path scale-free test:
      Large N (>=50): OLS log-log regression — strict R2_pl > R2_exp
      Small N (<50) : Hill MLE for gamma + monotone P(k) + R2_pl > 0.75
                      (OLS R2 comparison is statistically unreliable with <6
                       distinct k values; Hill MLE is the standard estimator
                       used in network science for small populations)
    """
    n_nodes = len(degrees)
    k_vals, counts = np.unique(degrees[degrees > 0], return_counts=True)
    pmf   = counts / len(degrees[degrees > 0])
    log_k = np.log(k_vals.astype(float))
    log_p = np.log(pmf + 1e-12)

    sl_pl,  ic_pl,  r_pl,  *_ = stats.linregress(log_k, log_p)
    sl_exp, ic_exp, r_exp, *_ = stats.linregress(k_vals.astype(float), log_p)
    gamma_ols = -sl_pl
    r2_pl     = r_pl  ** 2
    r2_exp    = r_exp ** 2
    gamma_hill = _hill_mle(degrees)
    monotone   = all(pmf[i] >= pmf[i+1] for i in range(len(pmf)-1))

    fitted_pl = np.exp(ic_pl) * (k_vals.astype(float) ** sl_pl)
    fitted_pl = fitted_pl / (fitted_pl.sum() + 1e-12)
    _, ks_p   = stats.ks_2samp(pmf, fitted_pl)

    small_n = n_nodes < 50

    print(f"\n{chr(8212)*55}\n  {label}\n{chr(8212)*55}")
    print(f"  Degree range       : [{degrees.min()}, {degrees.max()}]")
    print(f"  Mean +/- Std       : {degrees.mean():.2f} +/- {degrees.std():.2f}")
    print(f"  gamma (OLS log-log): {gamma_ols:.3f}")
    if small_n:
        print(f"  gamma (Hill MLE)   : {gamma_hill:.3f}  << used for small-N verdict")
    print(f"  R2 power-law       : {r2_pl:.4f}")
    print(f"  R2 exponential     : {r2_exp:.4f}")
    print(f"  KS p-value (PL)    : {ks_p:.4f}  (p>0.05 = good PL fit)")
    if small_n:
        print(f"  Monotone P(k)      : {monotone}")
        print(f"  Method             : Hill MLE (N={n_nodes} < 50; OLS unreliable with <6 k-values)")

    if small_n:
        # Hill MLE criterion for small-N graphs
        is_sf = 2.0 < gamma_hill < 3.5 and monotone and r2_pl > 0.75
    else:
        is_sf = r2_pl > r2_exp and 2 < gamma_ols < 3 and ks_p > 0.05

    if is_sf:
        print("  Verdict            : SCALE-FREE")
    else:
        reasons = []
        if small_n:
            if not (2.0 < gamma_hill < 3.5): reasons.append(f"Hill gamma={gamma_hill:.2f} outside (2,3.5)")
            if not monotone:                  reasons.append("P(k) not monotone decreasing")
            if r2_pl <= 0.75:                 reasons.append(f"R2_pl={r2_pl:.3f} < 0.75")
        else:
            if r2_pl <= r2_exp:  reasons.append(f"exp fits better (R2_exp={r2_exp:.3f} > R2_pl={r2_pl:.3f})")
            if not (2<gamma_ols<3): reasons.append(f"gamma={gamma_ols:.2f} outside (2,3)")
            if ks_p <= 0.05:     reasons.append(f"KS rejects PL (p={ks_p:.4f})")
        print(f"  Reason(s)          : {'; '.join(reasons)}")
        print("  Verdict            : NOT SCALE-FREE")

    gamma_report = gamma_hill if small_n else gamma_ols
    return gamma_report, r2_pl, r2_exp, ks_p


def plot_degree_distribution(pat_deg, gene_deg, title_prefix="Graph"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, degs, lbl in zip(
        axes,
        [pat_deg, gene_deg],
        ["Patient degree (# mutations per patient)",
         "Gene degree (# patients with mutation)"]
    ):
        k_vals, counts = np.unique(degs[degs > 0], return_counts=True)
        pmf = counts / len(degs[degs > 0])
        ax.loglog(k_vals, pmf, "o", color="steelblue", ms=6, label="Empirical P(k)")
        log_k, log_p = np.log(k_vals.astype(float)), np.log(pmf + 1e-12)
        sl_pl, ic_pl, *_ = stats.linregress(log_k, log_p)
        ax.loglog(k_vals, np.exp(ic_pl + sl_pl * log_k), "r--", lw=2,
                  label=f"Power-law  gamma={-sl_pl:.2f}")
        sl_ex, ic_ex, *_ = stats.linregress(k_vals.astype(float), log_p)
        ax.loglog(k_vals, np.exp(ic_ex + sl_ex * k_vals.astype(float)), "g:", lw=2,
                  label="Exponential")
        ax.set_xlabel("Degree k"); ax.set_ylabel("P(k)")
        ax.set_title(f"{title_prefix}\n{lbl}")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax.text(0.97, 0.97, "Scale-free => straight line on log-log",
                transform=ax.transAxes, fontsize=7.5, va="top", ha="right", color="gray",
                bbox=dict(fc="white", alpha=0.7, ec="lightgray"))
    plt.suptitle(f"Degree Distribution - {title_prefix}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig("V15_scalefree_degree_dist.png", dpi=150, bbox_inches="tight")
    plt.show()


# ── Run analysis ───────────────────────────────────────────────────────────────
print("=" * 65)
print("SCALE-FREE ANALYSIS - Original Bipartite Heterogeneous Graph")
print("=" * 65)
print("""
  Graph type  : Bipartite HeteroData  (Gene <-> Patient)
  Patient nodes : [Gender, Race, Age_norm]  (N ~800+)
  Gene nodes    : one-hot identity          (N = 20)
  Edges         : Gene->Patient if mutation bit == 1
""")

_ag = to_dev(construct_bipartite_heterograph(train_val_df))
_pd, _gd = get_degree_sequences(_ag)
_g_p, _r2pl_p, _r2ex_p, _ks_p = fit_powerlaw(_pd,  "Patient-side degrees")
_g_g, _r2pl_g, _r2ex_g, _ks_g = fit_powerlaw(_gd, "Gene-side degrees")

print(f"""
FORMAL VERDICT
--------------
Scale-free iff P(k) ~ k^(-gamma), 2<gamma<3, R2_PL > R2_exp

Patient side : gamma={_g_p:.2f}  R2_PL={_r2pl_p:.3f}  R2_exp={_r2ex_p:.3f}
Gene side    : gamma={_g_g:.2f}  R2_PL={_r2pl_g:.3f}  R2_exp={_r2ex_g:.3f}

WHY IT FAILS:
  1. Patient degree hard-capped at NUM_GENES=20 -> no heavy tail
  2. Gene frequencies Poisson-like, not power-law
  3. No preferential attachment (static mutation matrix)

-> APPLYING BA REWIRING BELOW
""")
plot_degree_distribution(_pd, _gd, title_prefix="V12 Original Graph")


SCALE-FREE ANALYSIS - Original Bipartite Heterogeneous Graph

  Graph type  : Bipartite HeteroData  (Gene <-> Patient)
  Patient nodes : [Gender, Race, Age_norm]  (N ~800+)
  Gene nodes    : one-hot identity          (N = 20)
  Edges         : Gene->Patient if mutation bit == 1


———————————————————————————————————————————————————————
  Patient-side degrees
———————————————————————————————————————————————————————
  Degree range       : [0, 17]
  Mean +/- Std       : 2.27 +/- 1.49
  gamma (OLS log-log): 2.125
  R2 power-law       : 0.7303
  R2 exponential     : 0.6229
  KS p-value (PL)    : 0.9801  (p>0.05 = good PL fit)
  Verdict            : SCALE-FREE

———————————————————————————————————————————————————————
  Gene-side degrees
———————————————————————————————————————————————————————
  Degree range       : [18, 322]
  Mean +/- Std       : 76.25 +/- 82.92
  gamma (OLS log-log): 0.134
  gamma (Hill MLE)   : 1.962  << used for small-N verdict
  R2 power-law       : 0.2005
  R2 exponential 

## 4b. Scale-Free Graph Construction — Dual BA Layers

**Patient-Patient layer (N~800)** — Sequential Barabási-Albert:
> Each patient enters one at a time, sorted by co-mutation seed degree (high-mutation patients enter first). Each new patient attaches to `ba_m=2` existing patients with probability proportional to accumulated degree — the "rich-get-richer" mechanism that yields P(k) ~ k⁻³.

**Gene-Gene layer (N=20)** — Sequential Barabási-Albert seeded by mutation frequency:
> Genes are added in mutation-frequency order (IDH1 first). Each new gene attaches to `ba_m_gene=2` existing genes with probability proportional to their mutation frequency. High-frequency genes like IDH1 and TP53 naturally become hubs. With only 20 genes the degree range is inherently narrow, so the BA structure is used for its biological relevance (hub genes co-mutate more) rather than as a strict power-law claim.

**Scale-free test:** OLS log-log for large N (≥50); Hill MLE for small N (<50).


In [ ]:
def construct_scalefree_bipartite_heterograph(df, ba_m=2, seed=42):
    """
    Full dual scale-free heterograph:

    Edges:
      Gene    -> mutates    -> Patient  (bipartite, unchanged)
      Patient -> mutated_by -> Gene     (reverse bipartite, unchanged)
      Patient -> cooccurs   -> Patient  (sequential BA: gamma ~2.2, heavy tail)
      Gene    -> coexists   -> Gene     (sequential BA: seeded by mutation frequency)

    Why two different algorithms:
      PP (N~800): true sequential Barabasi-Albert — each new node attaches to m
        existing nodes proportional to degree. Proven P(k)~k^(-3) for large N.
        The batch variant fails because all nodes enter simultaneously, preventing
        the rich-get-richer dynamic that creates a genuine heavy tail.

      GG (N=20): genes enter in mutation-frequency order. Each new gene attaches
        to ba_m_gene=2 existing genes proportional to mutation frequency + degree.
        IDH1, TP53, ATRX naturally become hubs. Same mechanism as PP layer.
    """
    rng     = np.random.default_rng(seed)
    mut_mat = df[gene_columns].values.astype(int)
    n_p     = len(df)

    # ── Standard bipartite ───────────────────────────────────────
    graph = HeteroData()
    # ── Patient feature encoding (correct) ────────────────────────────────
    # Gender      : binary 0/1          -> 1 feature (already correct)
    # Race        : nominal {0,1,2,3}   -> one-hot 4 features
    #               (raw int encoding is wrong: implies White<Asian<Black<Other)
    # Age_at_diag : continuous float    -> StandardScaler normalize -> 1 feature
    # Total: Patient.x = [Gender | Race_0 Race_1 Race_2 Race_3 | Age_norm]  shape [N, 6]
    scaler   = StandardScaler()
    age_norm = scaler.fit_transform(df[['Age_at_diagnosis']].values.astype(float))  # [N,1]
    gender   = df[['Gender']].values.astype(float)                                  # [N,1]
    race_ohe = np.zeros((len(df), 4), dtype=float)
    for i, rv in enumerate(df['Race'].values.astype(int)):
        if 0 <= rv < 4:
            race_ohe[i, rv] = 1.0                                                   # [N,4]
    pat_feats = np.hstack([gender, race_ohe, age_norm])                             # [N,6]
    graph["Patient"].x = torch.tensor(pat_feats, dtype=torch.float)
    graph["Patient"].y = torch.tensor(df["Grade"].values, dtype=torch.long)
    graph["Gene"].x    = torch.eye(NUM_GENES, dtype=torch.float)

    src_genes, dst_pats = [], []
    for p_idx, row in enumerate(mut_mat):
        for g_idx in np.where(row == 1)[0]:
            src_genes.append(g_idx); dst_pats.append(p_idx)

    graph[("Gene",   "mutates",   "Patient")].edge_index = torch.tensor([src_genes, dst_pats ], dtype=torch.long)
    graph[("Patient","mutated_by","Gene"   )].edge_index = torch.tensor([dst_pats,  src_genes], dtype=torch.long)

    # ── Patient-Patient: TRUE sequential Barabasi-Albert ─────────
    # Patients are added one at a time (sorted by co-mutation seed degree so
    # high-mutation patients enter early and become natural hubs).
    # Each new patient attaches to ba_m existing patients with probability
    # proportional to their accumulated degree + seed degree.
    # This is the canonical BA model proven to yield P(k) ~ k^(-3).
    # gamma empirically lands in (2.1, 2.5) for N~800, ba_m=2.

    seed_deg = np.zeros(n_p, dtype=float)
    for g in range(NUM_GENES):
        carriers = np.where(mut_mat[:, g] == 1)[0]
        for i in range(len(carriers)):
            for j in range(i+1, min(i+50, len(carriers))):
                a, b = int(carriers[i]), int(carriers[j])
                seed_deg[a] += 1; seed_deg[b] += 1
    seed_deg = np.maximum(seed_deg, 1.0)

    order   = np.argsort(-seed_deg)        # high seed degree enters first
    degree  = np.zeros(n_p, dtype=float)
    pp_edge_set = set()

    # Seed clique: first (ba_m+1) patients connect to each other
    for i in range(ba_m + 1):
        for j in range(i + 1, ba_m + 1):
            a, b = int(order[i]), int(order[j])
            pp_edge_set.add((min(a,b), max(a,b)))
            degree[a] += 1; degree[b] += 1

    # Sequential attachment
    for step in range(ba_m + 1, n_p):
        new_node = int(order[step])
        existing = order[:step]
        ed       = degree[existing] + seed_deg[existing]
        probs    = ed / ed.sum()
        for t in rng.choice(existing, size=min(ba_m, len(existing)),
                             replace=False, p=probs):
            a, b = min(new_node, int(t)), max(new_node, int(t))
            pp_edge_set.add((a, b))
            degree[new_node] += 1; degree[int(t)] += 1

    if pp_edge_set:
        pp_s, pp_d = zip(*pp_edge_set)
        # Store undirected as bidirectional for GNN message passing
        graph[("Patient","cooccurs","Patient")].edge_index = torch.tensor(
            [list(pp_s)+list(pp_d), list(pp_d)+list(pp_s)], dtype=torch.long)
    else:
        graph[("Patient","cooccurs","Patient")].edge_index = torch.zeros(2, 0, dtype=torch.long)

    # ── Gene-Gene: Sequential Barabasi-Albert seeded by mutation frequency ──
    # Genes enter in mutation-frequency order (most mutated first).
    # Each new gene attaches to ba_m_gene existing genes with probability
    # proportional to their mutation frequency + accumulated degree.
    # High-frequency genes (IDH1, TP53, ATRX) naturally become hubs.
    # With N=20 the distribution is narrow but the hub structure is
    # biologically meaningful: co-mutation hubs reflect driver gene co-occurrence.
    ba_m_gene = 2
    gene_mut_freq = np.maximum(mut_mat.sum(axis=0).astype(float), 1.0)  # [NUM_GENES]
    freq_order    = np.argsort(-gene_mut_freq)   # most mutated gene enters first
    gg_degree     = np.zeros(NUM_GENES, dtype=float)
    gg_edge_set   = set()

    # Seed clique among first (ba_m_gene+1) genes
    for i in range(ba_m_gene + 1):
        for j in range(i + 1, ba_m_gene + 1):
            a, b = int(freq_order[i]), int(freq_order[j])
            gg_edge_set.add((min(a, b), max(a, b)))
            gg_degree[a] += 1; gg_degree[b] += 1

    # Sequential attachment: each remaining gene attaches to ba_m_gene existing genes
    for step in range(ba_m_gene + 1, NUM_GENES):
        new_gene = int(freq_order[step])
        existing = freq_order[:step]
        ed       = gg_degree[existing] + gene_mut_freq[existing]
        probs    = ed / ed.sum()
        for t in rng.choice(existing, size=min(ba_m_gene, len(existing)),
                             replace=False, p=probs):
            a, b = min(new_gene, int(t)), max(new_gene, int(t))
            gg_edge_set.add((a, b))
            gg_degree[new_gene] += 1; gg_degree[int(t)] += 1

    if gg_edge_set:
        gg_s, gg_d = zip(*gg_edge_set)
        graph[("Gene","coexists","Gene")].edge_index = torch.tensor(
            [list(gg_s)+list(gg_d), list(gg_d)+list(gg_s)], dtype=torch.long)
    else:
        graph[("Gene","coexists","Gene")].edge_index = torch.zeros(2, 0, dtype=torch.long)

    return graph


# ── Verify both scale-free layers ────────────────────────────────
print("=" * 65)
print("SCALE-FREE VERIFICATION — BA Graph Layers")
print("=" * 65)

_sf_g  = construct_scalefree_bipartite_heterograph(train_val_df, ba_m=2)
_pp_ei = _sf_g[("Patient","cooccurs","Patient")].edge_index.cpu()
_gg_ei = _sf_g[("Gene","coexists","Gene")].edge_index.cpu()
# For PP: take one direction only (first half of bidirectional edges)
_n_pp  = _pp_ei.shape[1] // 2
# Both PP and GG are stored bidirectionally; count all edges and halve for true degree.
_pp_d  = np.bincount(_pp_ei[0].numpy(), minlength=_sf_g["Patient"].x.shape[0]) // 2
_gg_d  = np.bincount(_gg_ei[0].numpy(), minlength=NUM_GENES) // 2

print("\nPatient-Patient (sequential BA, N~800):")
_g_pp, _r2pl_pp, _r2ex_pp, _ks_pp = fit_powerlaw(_pp_d, "Patient-Patient (seq BA)")

print("\nGene-Gene (sequential BA, N=20):")
_g_gg, _r2pl_gg, _r2ex_gg, _ks_gg = fit_powerlaw(_gg_d, "Gene-Gene (seq BA)")

# Top hub genes
_hub_gene = gene_columns[int(np.argmax(_gg_d))]
_gg_top5  = sorted(zip(gene_columns, _gg_d), key=lambda x: -x[1])[:5]

print(f"""
SUMMARY
  Patient-Patient  gamma={_g_pp:.3f}  R2_PL={_r2pl_pp:.4f}  R2_exp={_r2ex_pp:.4f}
  Gene-Gene        gamma={_g_gg:.3f}  R2_PL={_r2pl_gg:.4f}  R2_exp={_r2ex_gg:.4f}
  Patient-Patient edges (undirected) : {_pp_ei.shape[1]//2}
  Gene-Gene edges (undirected)       : {_gg_ei.shape[1]//2}
  Gene-Patient edges                 : {_sf_g[("Gene","mutates","Patient")].edge_index.shape[1]}
  Hub gene: {_hub_gene} (degree {_gg_d.max()})
  Top-5 genes by GG degree: {_gg_top5}
""")

# ── Plot both degree distributions ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, d, label, color, note in [
    (axes[0], _pp_d, "Patient-Patient (sequential BA)", "steelblue",
     f"N={len(_pp_d)} | ba_m=2 | sequential BA"),
    (axes[1], _gg_d, "Gene-Gene (sequential BA)", "darkorange",
     f"N={NUM_GENES} | seq BA | ba_m_gene=2"),
]:
    k, c = np.unique(d[d > 0], return_counts=True)
    pmf  = c / c.sum()
    ax.loglog(k, pmf, "o", color=color, ms=9, alpha=0.90,
               markeredgecolor="white", markeredgewidth=0.6, zorder=4, label="Empirical P(k)")
    sl_pl, ic_pl, r_pl, *_ = stats.linregress(np.log(k.astype(float)), np.log(pmf + 1e-12))
    sl_ex, ic_ex, r_ex, *_ = stats.linregress(k.astype(float), np.log(pmf + 1e-12))
    k_sm = np.linspace(k.min(), k.max(), 100)
    ax.loglog(k_sm, np.exp(ic_pl + sl_pl * np.log(k_sm)), "r--", lw=2.2,
               label=f"Power-law  gamma_OLS={-sl_pl:.2f}  R²={r_pl**2:.3f}")
    ax.loglog(k_sm, np.exp(ic_ex + sl_ex * k_sm), "g:", lw=1.8,
               label=f"Exponential  R²={r_ex**2:.3f}", alpha=0.7)
    ax.set_xlabel("Degree k", fontsize=10); ax.set_ylabel("P(k)", fontsize=10)
    ax.set_title(f"{label}\nLog-Log Degree Distribution", fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.text(0.04, 0.06, note, transform=ax.transAxes, fontsize=7.5,
             va="bottom", ha="left", color="gray",
             bbox=dict(fc="white", alpha=0.75, ec="lightgray"))
    hill_g = _hill_mle(d)
    is_sf  = fit_powerlaw(d)[0]  # returns gamma
    vcol = "#27ae60" if 2.0 < hill_g < 3.5 else "#e74c3c"
    verdict = "SCALE-FREE" if 2.0 < hill_g < 3.5 else "NOT SF"
    ax.text(0.97, 0.97, f"{verdict}\ngamma_Hill={hill_g:.2f}",
             transform=ax.transAxes, fontsize=10, fontweight="bold",
             va="top", ha="right", color=vcol)

plt.suptitle("Scale-Free Verification — Dual BA Layers\n"
             "Sequential BA — Patient-Patient and Gene-Gene", fontsize=12)
plt.tight_layout()
plt.savefig("V16_scalefree_both_layers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: V16_scalefree_both_layers.png")


## 4c. Graph Visualisation — Full Heterogeneous Scale-Free Structure

The plot below shows a **subgraph** sampled from the training set to make the structure
legible:
- 20 Gene nodes (left column, orange) — size proportional to mutation frequency
- 40 Patient nodes (right cluster, blue=Grade0, red=Grade1) — size proportional to total incoming Gene attention
- **Black edges**: Gene-Patient (bipartite mutation edges)
- **Orange edges**: Gene-Gene co-mutation (new BA layer)
- **Light blue edges**: Patient-Patient co-occurrence (existing BA layer)


In [ ]:
import networkx as nx
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

def visualize_scalefree_graph(df, n_patients=40, seed=42):
    """
    Subgraph visualisation of the full heterogeneous scale-free graph.
    Samples n_patients patients stratified by grade.
    """
    rng   = np.random.default_rng(seed)
    grade = df['Grade'].values
    idx0  = rng.choice(np.where(grade == 0)[0], n_patients // 2, replace=False)
    idx1  = rng.choice(np.where(grade == 1)[0], n_patients // 2, replace=False)
    sub_idx = np.sort(np.concatenate([idx0, idx1]))
    sub_df  = df.iloc[sub_idx].reset_index(drop=True)

    g = construct_scalefree_bipartite_heterograph(sub_df, ba_m=2, seed=seed)
    mut_mat = sub_df[gene_columns].values.astype(int)

    # --- Build networkx graph ----------------------------------------
    G = nx.Graph()
    gene_nodes = [f"G_{gname}" for gname in gene_columns]
    pat_nodes  = [f"P_{i}" for i in range(len(sub_df))]

    G.add_nodes_from(gene_nodes, ntype='gene')
    G.add_nodes_from(pat_nodes,  ntype='patient')

    # Gene-Patient edges
    gp_ei = g[('Gene','mutates','Patient')].edge_index.cpu().numpy()
    gp_edges = [(f"G_{gene_columns[gi]}", f"P_{pi}") for gi, pi in zip(gp_ei[0], gp_ei[1])]
    G.add_edges_from(gp_edges, etype='bipartite')

    # Gene-Gene edges
    gg_ei = g[('Gene','coexists','Gene')].edge_index.cpu().numpy()
    gg_edges = [(f"G_{gene_columns[gi]}", f"G_{gene_columns[gj]}")
                for gi, gj in zip(gg_ei[0], gg_ei[1]) if gi < gj]
    G.add_edges_from(gg_edges, etype='gene_gene')

    # Patient-Patient edges
    pp_ei = g[('Patient','cooccurs','Patient')].edge_index.cpu().numpy()
    pp_edges = [(f"P_{pi}", f"P_{pj}") for pi, pj in zip(pp_ei[0], pp_ei[1]) if pi < pj]
    G.add_edges_from(pp_edges, etype='pat_pat')

    # --- Layout: genes on left spine, patients in right cluster ------
    pos = {}
    for i, gn in enumerate(gene_nodes):
        pos[gn] = (-2.0, 1 - 2 * i / (len(gene_nodes) - 1))

    # Patient positions: spring layout in right half
    pat_sub = nx.subgraph(G, pat_nodes)
    pat_pos_spring = nx.spring_layout(pat_sub, seed=seed, k=0.5)
    for pn, (x, y) in pat_pos_spring.items():
        pos[pn] = (x * 1.5 + 2.5, y * 1.5)

    # --- Node sizes and colours ───────────────────────────────────
    gene_mut_freq = mut_mat.sum(axis=0)   # [20]
    gene_sizes  = 200 + 600 * (gene_mut_freq / gene_mut_freq.max())

    pat_total_deg = np.array([G.degree(f"P_{i}") for i in range(len(sub_df))])
    pat_sizes     = 80 + 300 * (pat_total_deg / max(pat_total_deg.max(), 1))
    pat_colors    = ['#c0392b' if sub_df.iloc[i]['Grade'] == 1 else '#2980b9'
                     for i in range(len(sub_df))]

    # --- Draw ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(22, 13))
    ax.set_facecolor('#f8f9fa')
    ax.set_title("Heterogeneous Scale-Free Graph — Subgraph Visualisation\n"
                 f"(20 genes + {n_patients} patients sampled from training set)",
                 fontsize=13, pad=15)

    # Draw PP edges (light blue, thin)
    nx.draw_networkx_edges(G, pos, edgelist=pp_edges, ax=ax,
                            edge_color='#74b9ff', alpha=0.25, width=0.6,
                            style='solid', arrows=False)
    # Draw GG edges (orange, medium)
    nx.draw_networkx_edges(G, pos, edgelist=gg_edges, ax=ax,
                            edge_color='#e67e22', alpha=0.70, width=1.8,
                            style='solid', arrows=False)
    # Draw GP edges (dark gray, thin)
    nx.draw_networkx_edges(G, pos, edgelist=gp_edges, ax=ax,
                            edge_color='#636e72', alpha=0.15, width=0.5,
                            style='solid', arrows=False)

    # Draw patient nodes — Grade 0 (blue), Grade 1 (red), thick border for clarity
    nx.draw_networkx_nodes(G, pos, nodelist=pat_nodes, ax=ax,
                            node_color=pat_colors, node_size=list(pat_sizes),
                            alpha=0.92, linewidths=1.2,
                            edgecolors=['#1a5276' if sub_df.iloc[i]['Grade']==0 else '#922b21'
                                        for i in range(len(sub_df))])
    # Draw gene nodes
    nx.draw_networkx_nodes(G, pos, nodelist=gene_nodes, ax=ax,
                            node_color='#e67e22', node_size=list(gene_sizes),
                            alpha=0.95, linewidths=0.6, edgecolors='#d35400')
    # Gene labels
    gene_label_dict = {gn: gn.replace('G_','') for gn in gene_nodes}
    nx.draw_networkx_labels(G, pos, labels=gene_label_dict, ax=ax,
                             font_size=7.5, font_weight='bold', font_color='#2d3436')

    # Label every patient node with Grade (large) and abbreviated clinical info
    # Format: "0" or "1" with age shown for hub patients
    grade_labels = {f"P_{i}": str(int(sub_df.iloc[i]['Grade']))
                    for i in range(len(sub_df))}
    nx.draw_networkx_labels(G, pos, labels=grade_labels, ax=ax,
                             font_size=8, font_weight='bold', font_color='white')
    # For top-6 hub patients also annotate Age just outside the node
    top_hub_idx = np.argsort(pat_total_deg)[::-1][:6]
    gender_map  = {0: 'F', 1: 'M'}
    race_map    = {0: 'Wht', 1: 'Blk', 2: 'Asn', 3: 'Nat'}
    for i in top_hub_idx:
        pn = f"P_{i}"
        row = sub_df.iloc[i]
        age  = int(row['Age_at_diagnosis'])
        gen  = gender_map.get(int(row['Gender']), '?')
        race = race_map.get(int(row['Race']), '?')
        x, y = pos[pn]
        ax.text(x, y - 0.18, f"{gen}/{race}/{age}y",
                fontsize=6.5, ha='center', va='top', color='#2d3436',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.75, ec='none'))

    # Legend
    legend_elements = [
        mpatches.Patch(color='#e67e22', alpha=0.95, label='Gene node (size = mutation freq)'),
        mpatches.Patch(color='#2980b9', alpha=0.92, label='Patient Grade 0  [label: grade | hub: Gender/Race/Age]'),
        mpatches.Patch(color='#c0392b', alpha=0.92, label='Patient Grade 1  [label: "1"]'),
        mlines.Line2D([], [], color='#636e72', lw=1.2, alpha=0.6, label='Gene–Patient mutation edge'),
        mlines.Line2D([], [], color='#e67e22', lw=2,   alpha=0.8, label='Gene–Gene co-mutation BA'),
        mlines.Line2D([], [], color='#74b9ff', lw=1.5, alpha=0.6, label='Patient–Patient co-occurrence BA'),
    ]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=9,
               framealpha=0.92, edgecolor='#dfe6e9')

    # Statistics box
    n_g0 = int((sub_df['Grade']==0).sum()); n_g1 = int((sub_df['Grade']==1).sum())
    stats_txt = (
        f"Nodes: {NUM_GENES} genes  +  {n_patients} patients\n"
        f"  Grade 0: {n_g0}  |  Grade 1: {n_g1}\n"
        f"Patient features: Gender / Race (W/A/B/O) / Age\n"
        f"Gene-Patient edges : {len(gp_edges)}\n"
        f"Gene-Gene edges    : {len(gg_edges)}\n"
        f"Patient-Patient edges: {len(pp_edges)}"
    )
    ax.text(0.01, 0.97, stats_txt, transform=ax.transAxes,
             fontsize=8.5, va='top', ha='left',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85,
                       edgecolor='#b2bec3'))

    ax.axis('off')
    plt.tight_layout()
    plt.savefig('V16_graph_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: V16_graph_visualization.png")
    print(f"  Gene-Gene edges     : {len(gg_edges)}")
    print(f"  Patient-Patient edges: {len(pp_edges)}")
    print(f"  Gene-Patient edges  : {len(gp_edges)}")

# Install networkx if needed (usually pre-installed)
try:
    import networkx
    print(f"networkx {networkx.__version__} available")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'networkx', '-q'])

visualize_scalefree_graph(train_val_df, n_patients=40, seed=42)


## 5. Build Shared Evaluation Graphs

In [ ]:
# V13: ALL shared eval graphs now use the scale-free graph builder
# so evaluation is consistent with training across all pipelines.
val_graph  = to_dev(construct_scalefree_bipartite_heterograph(val_df))
test_graph = to_dev(construct_scalefree_bipartite_heterograph(test_df))
cgga_graph = to_dev(construct_scalefree_bipartite_heterograph(cgga_df))

print("Val  graph:", val_graph['Patient'].x.shape[0], "patients")
print("Test graph:", test_graph['Patient'].x.shape[0], "patients")
print("CGGA graph:", cgga_graph['Patient'].x.shape[0], "patients")

## 6. Model Definitions (7 architectures)

| # | Model | Core idea |
|---|-------|-----------|
| 1 | **HeteroGATv2** | Bidirectional GATv2 on Gene-Patient bipartite graph |
| 2 | **MOGAT** | Dual genomic+clinical paths, soft fusion gate |
| 3 | **HyperTMO** | Genes = hyperedges, patients = nodes |
| 4 | **RGCN** | Patient-patient co-mutation graph, 20 gene-typed relations |
| 5 | **VEGN** | Learned per-edge variant-effect weights |
| 6 | **FastHGTConv** | Heterogeneous Graph Transformer |
| 7 | **SGNN** | Chebyshev spectral filters on patient co-mutation adjacency |

In [ ]:
# ── 1. HeteroGATv2 ───────────────────────────────────────────
class HeteroGATv2(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin     = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.clin_skip = Linear(-1, hidden_dim)  # clinical bypass: Age / Race / Gender
        self.gg_conv = GCNConv(hidden_dim, hidden_dim)   # Gene-Gene co-mutation layer (new)
        self.g2p = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.p2g = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.p_skip = Linear(hidden_dim, hidden_dim); self.g_skip = Linear(hidden_dim, hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim)   # Patient-Patient co-occurrence layer
        self.clf = Linear(hidden_dim, out_dim)

    def forward(self, graph):
        ei    = graph.edge_index_dict
        x_pat = graph['Patient'].x
        h_clin = F.relu(self.clin_skip(x_pat))    # Age/Race/Gender preserved here
        hp = F.relu(self.p_lin(x_pat))
        hg = F.relu(self.g_lin(graph['Gene'].x))
        # Gene-Gene message passing (new): update gene embeddings before bipartite pass
        gg_ei = graph[('Gene','coexists','Gene')].edge_index.to(hg.device)
        if gg_ei.shape[1] > 0:
            hg = hg + F.relu(self.gg_conv(hg, gg_ei))
        hp = self.p_skip(hp) + self.g2p((hg, hp), ei[('Gene','mutates','Patient')])
        hg = self.g_skip(hg) + self.p2g((hp, hg), ei[('Patient','mutated_by','Gene')])
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)
        # Add clinical bypass after all gene/patient message-passing
        hp = F.dropout(F.leaky_relu(hp, 0.2), self.dr, training=self.training) + h_clin
        return self.clf(hp)

    def _get_enriched_gene_emb(self, graph):
        """Apply gg_conv so attention extraction mirrors the forward pass."""
        hg    = F.relu(self.g_lin(graph['Gene'].x))
        gg_ei = graph[('Gene','coexists','Gene')].edge_index.to(hg.device)
        if gg_ei.shape[1] > 0:
            hg = hg + F.relu(self.gg_conv(hg, gg_ei))
        return hg

    def get_attn_weights(self, graph):
        """Gene->Patient GATv2 attention (mean across heads), using enriched gene emb."""
        ei = graph.edge_index_dict
        hp = F.relu(self.p_lin(graph['Patient'].x))
        hg = self._get_enriched_gene_emb(graph)
        _, (eidx, alpha) = self.g2p((hg, hp), ei[('Gene','mutates','Patient')],
                                     return_attention_weights=True)
        return eidx, alpha.detach().mean(dim=-1)

    def get_all_heads_attn(self, graph):
        """Per-head Gene->Patient attention [E, H], using enriched gene emb."""
        ei = graph.edge_index_dict
        hp = F.relu(self.p_lin(graph['Patient'].x))
        hg = self._get_enriched_gene_emb(graph)
        _, (eidx, alpha) = self.g2p((hg, hp), ei[('Gene','mutates','Patient')],
                                     return_attention_weights=True)
        return eidx, alpha.detach()

    def get_p2g_attn_weights(self, graph):
        """Patient->Gene reverse GATv2 attention (mean across heads), using enriched gene emb."""
        ei    = graph.edge_index_dict
        hp    = F.relu(self.p_lin(graph['Patient'].x))
        hg    = self._get_enriched_gene_emb(graph)
        hp_upd = self.p_skip(hp) + self.g2p((hg, hp), ei[('Gene','mutates','Patient')])
        _, (eidx_r, alpha_r) = self.p2g((hp_upd, hg), ei[('Patient','mutated_by','Gene')],
                                         return_attention_weights=True)
        return eidx_r, alpha_r.detach().mean(dim=-1)


# ── 2. MOGAT ──────────────────────────────────────────────────────
class MOGAT(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.pg = Linear(-1, hidden_dim); self.gg = Linear(-1, hidden_dim)
        self.gg_conv = GCNConv(hidden_dim, hidden_dim)   # Gene-Gene co-mutation layer (new)
        self.gat = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.pc = Linear(-1, hidden_dim)
        self.mlp = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.gate = nn.Linear(hidden_dim * 2, 2)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim)   # Patient-Patient co-occurrence layer
        self.clf = Linear(hidden_dim, out_dim)

    def forward(self, graph):
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(self.pg(graph['Patient'].x))
        hgg = F.relu(self.gg(graph['Gene'].x))
        # Gene-Gene message passing (new): enrich gene embeddings before bipartite pass
        gg_ei = graph[('Gene','coexists','Gene')].edge_index.to(hgg.device)
        if gg_ei.shape[1] > 0:
            hgg = hgg + F.relu(self.gg_conv(hgg, gg_ei))
        hpg = F.dropout(F.leaky_relu(self.gat((hgg, hpg), e), 0.2), self.dr, training=self.training)
        hpc = self.mlp(F.relu(self.pc(graph['Patient'].x)))
        g   = torch.softmax(self.gate(torch.cat([hpg, hpc], -1)), dim=-1)
        hp  = g[:, :1] * hpg + g[:, 1:] * hpc
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)
        return self.clf(hp)

    def _get_enriched_gene_emb(self, graph):
        """Apply gg_conv so attention extraction mirrors the forward pass."""
        hgg   = F.relu(self.gg(graph['Gene'].x))
        gg_ei = graph[('Gene','coexists','Gene')].edge_index.to(hgg.device)
        if gg_ei.shape[1] > 0:
            hgg = hgg + F.relu(self.gg_conv(hgg, gg_ei))
        return hgg

    def get_attn_weights(self, graph):
        """Gene->Patient GATv2 attention (mean across heads), using enriched gene emb."""
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(self.pg(graph['Patient'].x))
        hgg = self._get_enriched_gene_emb(graph)
        _, (eidx, alpha) = self.gat((hgg, hpg), e, return_attention_weights=True)
        return eidx, alpha.detach().mean(dim=-1)

    def get_all_heads_attn(self, graph):
        """Per-head Gene->Patient attention [E, H], using enriched gene emb."""
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(self.pg(graph['Patient'].x))
        hgg = self._get_enriched_gene_emb(graph)
        _, (eidx, alpha) = self.gat((hgg, hpg), e, return_attention_weights=True)
        return eidx, alpha.detach()

    def get_fusion_gate(self, graph):
        """Per-patient fusion gate [N, 2] (genomic, clinical), using enriched gene emb."""
        e   = graph[('Gene','mutates','Patient')].edge_index
        hpg = F.relu(self.pg(graph['Patient'].x))
        hgg = self._get_enriched_gene_emb(graph)
        hpg_out = F.leaky_relu(self.gat((hgg, hpg), e), 0.2)
        hpc     = self.mlp(F.relu(self.pc(graph['Patient'].x)))
        gate_w  = torch.softmax(self.gate(torch.cat([hpg_out, hpc], -1)), dim=-1)
        return gate_w.detach()


# ── 3. HyperTMO ───────────────────────────────────────────────────
class HyperTMO(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, in_channels=6, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin     = nn.Linear(in_channels, hidden_dim)
        self.clin_skip = nn.Linear(in_channels, hidden_dim)  # clinical bypass: Age/Race/Gender
        self.g_lin     = nn.Linear(NUM_GENES, hidden_dim)
        self.hc1 = HypergraphConv(hidden_dim, hidden_dim, use_attention=True,
                                   heads=num_heads, dropout=dropout, concat=False)
        self.hc2 = HypergraphConv(hidden_dim, hidden_dim)
        self.skip = nn.Linear(hidden_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim)   # residual PP layer for cooccurs
        self.clf = nn.Linear(hidden_dim, out_dim)

    def forward(self, graph):
        xp = graph['Patient'].x; xg = graph['Gene'].x
        h_clin = F.relu(self.clin_skip(xp))    # clinical bypass: Age/Race/Gender
        ei  = graph[('Gene','mutates','Patient')].edge_index
        hei = torch.stack([ei[1], ei[0]], dim=0)
        hp = F.relu(self.p_lin(xp)); hg = F.relu(self.g_lin(xg)); s = self.skip(hp)
        hp = F.relu(self.hc1(hp, hei, hyperedge_attr=hg))
        hp = F.dropout(hp, self.dr, training=self.training)
        hp = self.bn(F.relu(self.hc2(hp, hei) + s))
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp.device)
        hp = hp + self.pp_conv(hp, pp_ei)  # one extra GCNConv layer
        return self.clf(F.dropout(hp, self.dr, training=self.training))


# ── 4. RGCN ───────────────────────────────────────────────────────
class RGCNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2,
                 in_channels=6, num_relations=20, **_):
        super().__init__()
        self.dr = dropout
        self.lin = nn.Linear(in_channels, hidden_dim)
        self.rc1 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.rc2 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.skip = nn.Linear(hidden_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim); self.clf = nn.Linear(hidden_dim, out_dim)

    def forward(self, graph):
        xp = graph['Patient'].x
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(xp.device)
        pp_et = torch.zeros(pp_ei.shape[1], dtype=torch.long, device=xp.device)
        h = F.relu(self.lin(xp)); s = self.skip(h)
        h = F.relu(self.rc1(h, pp_ei, pp_et))
        h = F.dropout(h, self.dr, training=self.training)
        h = self.bn(F.relu(self.rc2(h, pp_ei, pp_et) + s))
        return self.clf(F.dropout(h, self.dr, training=self.training))


# ── 5. VEGN ───────────────────────────────────────────────────────
class VEGNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2, **_):
        super().__init__()
        self.dr = dropout
        self.p_lin     = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.clin_skip = Linear(-1, hidden_dim)  # clinical bypass: Age/Race/Gender
        self.ve = nn.Sequential(nn.Linear(hidden_dim*2, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.p2g = GATv2Conv(hidden_dim, hidden_dim, heads=num_heads, concat=False, add_self_loops=False)
        self.skip = Linear(hidden_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.pp_conv = GCNConv(hidden_dim, hidden_dim)   # residual PP layer for cooccurs
        self.clf = Linear(hidden_dim, out_dim)

    def forward(self, graph):
        xp = graph['Patient'].x; xg = graph['Gene'].x
        h_clin = F.relu(self.clin_skip(xp))   # clinical bypass: Age/Race/Gender
        e_g2p = graph[('Gene','mutates','Patient')].edge_index
        e_p2g = graph[('Patient','mutated_by','Gene')].edge_index
        hp = F.relu(self.p_lin(xp)); hg = F.relu(self.g_lin(xg))
        sg, dp = e_g2p[0], e_g2p[1]
        wt = self.ve(torch.cat([hg[sg], hp[dp]], -1)).squeeze(-1)
        msg = hg[sg] * wt.unsqueeze(-1)
        agg = torch.zeros_like(hp); agg.scatter_add_(0, dp.unsqueeze(-1).expand_as(msg), msg)
        deg = torch.zeros(hp.shape[0], device=hp.device)
        deg.scatter_add_(0, dp, wt); deg = deg.clamp(min=1)
        agg = agg / deg.unsqueeze(-1)
        _ = self.p2g((hp, hg), e_p2g)
        hp2 = self.bn(F.leaky_relu(self.skip(hp) + agg, 0.2))
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(hp2.device)
        hp2 = hp2 + self.pp_conv(hp2, pp_ei)  # one extra GCNConv layer
        hp = F.dropout(hp, self.dr, training=self.training) + h_clin
        return self.clf(hp)


# ── 6. FastHGTConv ────────────────────────────────────────────────
# _HGT_META_SF includes the BA scale-free cooccurs relation.
# For standard graphs the cooccurs edges are absent (0 edges) so
# HGTConv simply produces zero contribution for that relation — safe.
_HGT_META = (['Patient','Gene'],
              [('Gene','mutates','Patient'),('Patient','mutated_by','Gene')])

_HGT_META_SF = (['Patient','Gene'],
                [('Gene','mutates','Patient'),('Patient','mutated_by','Gene'),
                 ('Patient','cooccurs','Patient'),('Gene','coexists','Gene')])  # V16: Gene-Gene layer added

# V16: FastHGTModel uses _HGT_META_SF (4 relations: g2p, p2g, pp-cooccurs, gg-coexists).
# When a standard bipartite graph is passed, HGTConv produces
# zero contribution for absent relations — mathematically safe.
class FastHGTModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2,
                 hgt_meta=None, **_):
        super().__init__()
        self.dr = dropout
        meta = hgt_meta if hgt_meta is not None else _HGT_META_SF
        self.p_lin     = Linear(-1, hidden_dim); self.g_lin = Linear(-1, hidden_dim)
        self.clin_skip = Linear(-1, hidden_dim)  # clinical bypass: Age/Race/Gender
        self.hgt1 = HGTConv(hidden_dim, hidden_dim, metadata=meta, heads=num_heads)
        self.hgt2 = HGTConv(hidden_dim, hidden_dim, metadata=meta, heads=num_heads)
        self.skip = Linear(hidden_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim); self.clf = Linear(hidden_dim, out_dim)

    def forward(self, graph):
        h_clin = F.relu(self.clin_skip(graph['Patient'].x))  # clinical bypass
        xd = {'Patient': F.relu(self.p_lin(graph['Patient'].x)),
              'Gene':    F.relu(self.g_lin(graph['Gene'].x))}
        sp = self.skip(xd['Patient']); ei = graph.edge_index_dict
        xd = {k: F.dropout(F.relu(v), self.dr, training=self.training)
              for k, v in self.hgt1(xd, ei).items()}
        xd = self.hgt2(xd, ei)
        hp = self.bn(F.relu(xd['Patient'] + sp))
        return self.clf(F.dropout(hp, self.dr, training=self.training) + h_clin)


# ── 7. SGNN ───────────────────────────────────────────────────────
class SGNNModel(nn.Module):
    def __init__(self, hidden_dim=32, out_dim=2, num_heads=4, dropout=0.2,
                 in_channels=6, K=3, **_):
        super().__init__()
        self.dr = dropout
        self.lin = nn.Linear(in_channels, hidden_dim)
        self.c1 = ChebConv(hidden_dim, hidden_dim, K=K)
        self.c2 = ChebConv(hidden_dim, hidden_dim, K=K)
        self.skip = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim); self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.clf = nn.Linear(hidden_dim, out_dim)

    def forward(self, graph):
        xp = graph['Patient'].x
        pp_ei = graph[('Patient','cooccurs','Patient')].edge_index.to(xp.device)
        h = F.relu(self.lin(xp)); s = self.skip(h)
        h = self.bn1(F.relu(self.c1(h, pp_ei)))
        h = F.dropout(h, self.dr, training=self.training)
        h = self.bn2(F.relu(self.c2(h, pp_ei) + s))
        return self.clf(F.dropout(h, self.dr, training=self.training))


# ── Registry ──────────────────────────────────────────────────────
MODEL_REGISTRY = [
    ('HeteroGATv2', HeteroGATv2,  {}),
    ('MOGAT',       MOGAT,        {}),
    ('HyperTMO',    HyperTMO,     {'in_channels': 6}),
    ('RGCN',        RGCNModel,    {'in_channels': 6, 'num_relations': NUM_GENES}),
    ('VEGN',        VEGNModel,    {}),
    ('FastHGTConv', FastHGTModel, {'hgt_meta': _HGT_META_SF}),
    ('SGNN',        SGNNModel,    {'in_channels': 6}),
]
print(f"Registered {len(MODEL_REGISTRY)} models:", [n for n,_,_ in MODEL_REGISTRY])

## 7. Focal Loss + Softened Class Weights

**Patient features** (updated): `Patient.x` = `[Gender | Race_0 Race_1 Race_2 Race_3 | Age_norm]` — shape `[N, 6]`

| Feature | Encoding | Reason |
|---------|----------|--------|
| Gender | Binary 0/1 (1 feature) | Natural binary |
| Race | One-hot {0,1,2,3} → 4 features | Nominal — no ordinal relationship |
| Age | StandardScaler z-score (1 feature) | Continuous, scale-sensitive |

**Fix**: penalty changed from `ratio` (≈1.38) to `sqrt(ratio)` (≈1.18).  
The FocalLoss `gamma=2` already down-weights easy examples; stacking a high class penalty over-corrects and inflates Grade-1 recall. Softening to square-root keeps a mild correction without making the decision boundary too aggressive toward Grade-1.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, weight=None):
        super().__init__()
        self.a, self.g, self.w = alpha, gamma, weight
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, weight=self.w, reduction='none')
        return (self.a * (1 - torch.exp(-ce)) ** self.g * ce).mean()

counts  = train_df['Grade'].value_counts().sort_index()   # [n_grade0, n_grade1]
# ── Inverse-frequency class weights (both classes weighted, not just minority) ──
# Grade-0 patients are the majority class; Grade-1 is the minority.
# Giving Grade-0 weight = n_grade1/total and Grade-1 weight = n_grade0/total
# ensures the loss treats every patient equally regardless of class size.
# This prevents the model from ignoring Grade-0 to chase Grade-1 recall.
n0, n1   = float(counts[0]), float(counts[1])
total    = n0 + n1
w0       = total / (2.0 * n0)   # < 1 when Grade-0 is majority
w1       = total / (2.0 * n1)   # > 1 when Grade-1 is minority
cw       = torch.tensor([w0, w1], dtype=torch.float).to(device)
criterion = FocalLoss(alpha=1, gamma=2, weight=cw)
print(f"Class weights: Grade-0={w0:.4f}  Grade-1={w1:.4f}  (inverse-freq, both balanced)")
print(f"  n_grade0={int(n0)}  n_grade1={int(n1)}  imbalance_ratio={n0/n1:.2f}:1")

## 8. Threshold Optimisation + Evaluation Helpers

**Fix**: `find_optimal_threshold()` searches `[0.30, 0.75]` for the threshold maximising **G-mean = √(recall₀ × recall₁)**. G-mean is maximised when both recalls are balanced, so Grade-1 recall can no longer climb arbitrarily high at Grade-0's expense.

**AUC fix**: `compute_metrics()` uses explicit `try/except` around `roc_auc_score` and returns `nan` for degenerate folds rather than crashing.

In [ ]:
# ── FIX: G-mean threshold search ─────────────────────────────────
def find_optimal_threshold(probs: np.ndarray, labels: np.ndarray,
                            low: float = 0.20, high: float = 0.80,
                            step: float = 0.005) -> float:
    """Return the decision threshold that maximises G-mean = sqrt(recall_0 * recall_1).
    Tie-breaking: among thresholds within 0.5% of best G-mean, prefer the one
    with smallest |recall_1 - recall_0| to avoid a lopsided recall profile.
    Search range widened to [0.20, 0.80] and step halved to 0.005."""
    best_th, best_gm, best_imb = 0.5, -1.0, 1.0
    for th in np.arange(low, high + step * 0.5, step):
        preds = (probs >= th).astype(int)
        r1  = recall_score(labels, preds, pos_label=1, zero_division=0)
        r0  = recall_score(labels, preds, pos_label=0, zero_division=0)
        gm  = np.sqrt(r0 * r1)
        imb = abs(r1 - r0)
        # Accept if strictly better G-mean, or near-equal G-mean with better balance
        if gm > best_gm + 0.005:
            best_gm = gm; best_th = float(th); best_imb = imb
        elif gm >= best_gm - 0.005 and imb < best_imb:
            best_th = float(th); best_imb = imb
    return best_th


def evaluate_model(model, graph, threshold: float = 0.5):
    """Return (preds, probs, labels) using a calibrated threshold."""
    model.eval()
    with torch.no_grad():
        logits = model(graph)
        probs  = F.softmax(logits, 1)[:, 1].cpu().numpy()
        labels = graph['Patient'].y.cpu().numpy()
    preds = (probs >= threshold).astype(int)
    return preds, probs, labels


def compute_metrics(preds: np.ndarray, probs: np.ndarray,
                     labels: np.ndarray) -> dict:
    """AUC fix: explicit try/except; also report per-class recall."""
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')   # single-class fold guard
    return {
        'auc':       auc,
        'accuracy':  accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall':    recall_score(labels, preds, zero_division=0),       # Grade-1 recall
        'recall_0':  recall_score(labels, preds, pos_label=0, zero_division=0),
        'f1':        f1_score(labels, preds, zero_division=0),
    }


# Storage containers
all_results  = []    # final test/CGGA results
cv_results   = []    # per-fold CV results
all_models   = {}    # {(model_name, pipeline): trained model}
all_studies  = {}    # {(model_name, pipeline): optuna study}
all_params   = {}    # {(model_name, pipeline): best_params}
all_thresholds = {}  # {(model_name, pipeline): threshold}
PIPELINES    = ['No Balancing', 'SMOTE', 'CTGAN', 'ROS']

## 9. Unified Training Function

In [ ]:
def train_and_evaluate(train_graph, val_graph, params, ModelClass,
                        fixed_kw=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                        seed=42):
    """Train model; return (best_val_auc, best_state, history)."""
    set_seed(seed)   # reset RNG before every model init+train for reproducibility
    kw = {'hidden_dim': params['hidden_dim'], 'out_dim': 2,
          'num_heads':  params['num_heads'],  'dropout': params['dropout']}
    if fixed_kw: kw.update(fixed_kw)

    model = ModelClass(**kw).to(device)
    try:
        with torch.no_grad(): _ = model(train_graph)
    except Exception: pass

    opt = torch.optim.AdamW(model.parameters(),
                              lr=params['lr'], weight_decay=params['weight_decay'])
    sch = CosineAnnealingLR(opt, T_max=max_epochs, eta_min=params['lr'] * 0.05)

    best_auc, ctr, best_state, history = 0.0, 0, None, []
    for _ in range(max_epochs):
        model.train(); opt.zero_grad()
        loss = criterion(model(train_graph), train_graph['Patient'].y)
        loss.backward(); opt.step(); sch.step()

        model.eval()
        with torch.no_grad():
            vp  = F.softmax(model(val_graph), 1)[:, 1].cpu().numpy()
            vl  = val_graph['Patient'].y.cpu().numpy()
        try:
            auc = roc_auc_score(vl, vp)
        except ValueError:
            auc = 0.0
        history.append(auc)

        if auc > best_auc:
            best_auc, ctr = auc, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            ctr += 1
            if ctr >= patience: break

    return best_auc, best_state, history

## 10. Optuna HPO (warm-up on train_df / val_df)

In [ ]:
def run_optuna(train_graph, val_graph, ModelClass, fixed_kw=None,
               n_trials=N_TRIALS, label=''):
    """Bayesian HPO; return (best_params, best_state, study)."""

    def objective(trial):
        set_seed(42)   # same RNG state for every trial → fair comparison
        params = {
            'hidden_dim':   trial.suggest_categorical('hidden_dim', [16, 32, 64, 128]),
            'num_heads':    trial.suggest_categorical('num_heads',  [2, 4, 8]),
            'dropout':      trial.suggest_float('dropout', 0.05, 0.30, step=0.05),
            'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        }
        auc, _, _ = train_and_evaluate(train_graph, val_graph, params, ModelClass, fixed_kw)
        return auc

    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    bp = study.best_params
    print(f"  [{label}] Best AUC={study.best_value:.4f}  params={bp}")
    _, best_state, _ = train_and_evaluate(train_graph, val_graph, bp, ModelClass, fixed_kw)
    return bp, best_state, study

## 11. 5-Fold Stratified Cross-Validation

Uses the **fixed best params** found by Optuna.  Each fold: augment training portion → build graph → train → find G-mean threshold on val → evaluate.  
Reports mean ± std across folds for AUC, Accuracy, Precision, Recall (Grade-1 & Grade-0), F1, Threshold.

In [ ]:
def apply_smote(fold_train_df):
    feat_cols = gene_columns + ['Gender','Race','Age_at_diagnosis']
    cat_idx   = [i for i,c in enumerate(feat_cols)
                  if c in gene_columns or c in ['Gender','Race']]
    sm = SMOTENC(categorical_features=cat_idx, random_state=42, k_neighbors=3)
    Xr, yr = sm.fit_resample(fold_train_df[feat_cols], fold_train_df['Grade'])
    df2 = pd.DataFrame(Xr, columns=feat_cols); df2['Grade'] = yr
    for c in gene_columns: df2[c] = df2[c].round().astype(int)
    df2['Race']   = df2['Race'].round().astype(int).clip(0, 3)
    df2['Gender'] = df2['Gender'].round().astype(int).clip(0, 1)
    return df2


def apply_ctgan(fold_train_df):
    meta = SingleTableMetadata()
    meta.detect_from_dataframe(fold_train_df)
    for col in categorical_columns + gene_columns:
        meta.update_column(column_name=col, sdtype='categorical')
    vc    = fold_train_df['Grade'].value_counts()
    n_gen = int(vc.max() - vc.min())
    if n_gen <= 0:
        return fold_train_df
    syn = CTGANSynthesizer(meta, epochs=100, batch_size=50, verbose=False, cuda=True, pac=10)
    syn._model_kwargs = {**getattr(syn, "_model_kwargs", {})}
    torch.manual_seed(42); np.random.seed(42)
    syn.fit(fold_train_df)
    cond  = Condition(num_rows=n_gen, column_values={'Grade': int(vc.idxmin())})
    extra = syn.sample_from_conditions(conditions=[cond])
    return pd.concat([fold_train_df, extra], ignore_index=True)


def run_5fold_cv(train_val_df, best_params, ModelClass, fixed_kw,
                  pipeline_name, model_name,
                  augment_fn=None,
                  graph_fn=None):
    """
    Run StratifiedKFold(5) with fixed best_params.

    FIX v12.1
    ---------
    augment_fn : callable(df) -> df  — replaces hardcoded name-based dispatch.
                 Accepts any balancing function (SMOTE / CTGAN / ROS / None).
    graph_fn   : callable(df) -> HeteroData — replaces hardcoded
                 construct_bipartite_heterograph so SF pipelines use the
                 scale-free graph builder throughout CV as well as HPO.
    """
    if graph_fn is None:
        graph_fn = construct_bipartite_heterograph

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_records = []

    for fold, (tr_idx, vl_idx) in enumerate(
            skf.split(train_val_df, train_val_df['Grade']), start=1):
        fold_tr = train_val_df.iloc[tr_idx].copy().reset_index(drop=True)
        fold_vl = train_val_df.iloc[vl_idx].copy().reset_index(drop=True)

        set_seed(42)

        # ── FIX: use augment_fn instead of name-based dispatch ────
        if augment_fn is not None:
            fold_tr = augment_fn(fold_tr)

        clear_pp_cache()
        # ── FIX: use graph_fn so SF pipelines build SF graphs ─────
        tr_g = to_dev(graph_fn(fold_tr))
        vl_g = to_dev(graph_fn(fold_vl))

        _, state, _ = train_and_evaluate(tr_g, vl_g, best_params, ModelClass, fixed_kw)

        kw = {'hidden_dim': best_params['hidden_dim'], 'out_dim': 2,
              'num_heads':  best_params['num_heads'],  'dropout': best_params['dropout']}
        kw.update(fixed_kw)
        fold_model = ModelClass(**kw).to(device)
        try:
            with torch.no_grad(): _ = fold_model(tr_g)
        except: pass
        fold_model.load_state_dict(state)

        _, probs_v, labels_v = evaluate_model(fold_model, vl_g, threshold=0.5)
        th = find_optimal_threshold(probs_v, labels_v)
        preds_v = (probs_v >= th).astype(int)

        m = compute_metrics(preds_v, probs_v, labels_v)
        m.update({'fold': fold, 'threshold': th,
                  'Model': model_name, 'Pipeline': pipeline_name})
        fold_records.append(m)
        print(f"    Fold {fold}/5 | AUC={m['auc']:.4f} "
              f"R1={m['recall']:.3f} R0={m['recall_0']:.3f} "
              f"F1={m['f1']:.4f} th={th:.2f}")

    clear_pp_cache()
    return pd.DataFrame(fold_records)


## 12. Pipeline Runner (shared by all 3 pipelines)

In [ ]:
def run_pipeline(pipeline_name, train_graph_hpo,
                  augment_fn=None, graph_fn=None):
    """
    For each model in MODEL_REGISTRY:
      1. Optuna HPO on provided train_graph_hpo
      2. 5-Fold CV on train_val_df with best params
      3. Final model on full train_val (augmented if augment_fn given)
      4. Evaluate on TCGA test + CGGA

    FIX v12.1
    ---------
    graph_fn : callable(df) -> HeteroData  (default: construct_bipartite_heterograph)
               Used for CV folds AND final model training so the graph structure
               is consistent with the HPO graph passed as train_graph_hpo.
               Pass construct_scalefree_bipartite_heterograph for SF pipelines.
    """
    if graph_fn is None:
        graph_fn = construct_bipartite_heterograph

    print(f"\n{'='*65}")
    print(f"PIPELINE: {pipeline_name}")
    print(f"{'='*65}")

    for mname, MCls, fkw in MODEL_REGISTRY:
        print(f"\n  -- {mname} --")

        # ── Step 1: Optuna HPO ────────────────────────────────────
        bp, hpo_state, study = run_optuna(
            train_graph_hpo, val_graph, MCls, fkw,
            label=f"{mname}/{pipeline_name}")
        all_studies[(mname, pipeline_name)] = study
        all_params[(mname, pipeline_name)]  = bp

        # ── Step 2: 5-Fold CV ─────────────────────────────────────
        print(f"  5-Fold CV with best params:")
        # FIX: pass augment_fn and graph_fn — no more name-based dispatch
        cv_df = run_5fold_cv(train_val_df, bp, MCls, fkw,
                              pipeline_name, mname,
                              augment_fn=augment_fn,
                              graph_fn=graph_fn)
        cv_results.append(cv_df)
        for col in ['auc','accuracy','precision','recall','recall_0','f1','threshold']:
            if col in cv_df.columns:
                print(f"    {col:12s}: {cv_df[col].mean():.4f} +/- {cv_df[col].std():.4f}")

        # ── Step 3: Final model on full train_val ─────────────────
        if augment_fn is not None:
            aug_df = augment_fn(train_val_df.copy())
        else:
            aug_df = train_val_df
        clear_pp_cache()
        # FIX: use graph_fn for consistency with HPO and CV
        full_tr_graph = to_dev(graph_fn(aug_df))

        _, final_state, _ = train_and_evaluate(full_tr_graph, val_graph, bp, MCls, fkw)

        kw = {'hidden_dim': bp['hidden_dim'], 'out_dim': 2,
              'num_heads':  bp['num_heads'],  'dropout': bp['dropout']}
        kw.update(fkw)
        final_model = MCls(**kw).to(device)
        try:
            with torch.no_grad(): _ = final_model(full_tr_graph)
        except: pass
        final_model.load_state_dict(final_state)

        th = float(cv_df['threshold'].mean())
        all_thresholds[(mname, pipeline_name)] = th
        print(f"  Final threshold (mean of CV folds) = {th:.3f}")

        # ── Step 4: Evaluate on test / CGGA ───────────────────────
        pdt, pbt, lbt = evaluate_model(final_model, test_graph,  threshold=th)
        pdc, pbc, lbc = evaluate_model(final_model, cgga_graph,  threshold=th)
        mt  = compute_metrics(pdt, pbt, lbt)
        mc_ = compute_metrics(pdc, pbc, lbc)
        print(f"  TCGA-Test  AUC={mt['auc']:.4f} R1={mt['recall']:.3f} R0={mt['recall_0']:.3f} F1={mt['f1']:.4f}")
        print(f"  CGGA       AUC={mc_['auc']:.4f} R1={mc_['recall']:.3f} R0={mc_['recall_0']:.3f} F1={mc_['f1']:.4f}")

        for ds, m, p, l in [('TCGA Test', mt, pbt, lbt), ('CGGA', mc_, pbc, lbc)]:
            rec = {'Model': mname, 'Pipeline': pipeline_name, 'Dataset': ds,
                   'threshold': th, 'probs': p, 'labels': l}
            rec.update(m)
            all_results.append(rec)
        all_models[(mname, pipeline_name)] = final_model
        clear_pp_cache()

    print(f"\n[{pipeline_name}] Done.")


## 13. Pipeline A — No Balancing

In [ ]:
# V13: Pipeline A uses scale-free graph builder
train_nb_graph = to_dev(construct_scalefree_bipartite_heterograph(train_df))
run_pipeline('No Balancing', train_graph_hpo=train_nb_graph, augment_fn=None,
             graph_fn=construct_scalefree_bipartite_heterograph)

## 14. Pipeline B — SMOTE

In [ ]:
feat_cols = gene_columns + ['Gender','Race','Age_at_diagnosis']
cat_idx   = [i for i,c in enumerate(feat_cols)
              if c in gene_columns or c in ['Gender','Race']]
smote   = SMOTENC(categorical_features=cat_idx, random_state=42, k_neighbors=3)
Xr, yr  = smote.fit_resample(train_df[feat_cols], train_df['Grade'])
train_smote_df = pd.DataFrame(Xr, columns=feat_cols); train_smote_df['Grade'] = yr
for c in gene_columns: train_smote_df[c] = train_smote_df[c].round().astype(int)
print("SMOTE HPO graph class dist:\n", train_smote_df['Grade'].value_counts())

# V13: Pipeline B uses scale-free graph builder
train_sm_graph = to_dev(construct_scalefree_bipartite_heterograph(train_smote_df))
run_pipeline('SMOTE', train_graph_hpo=train_sm_graph, augment_fn=apply_smote,
             graph_fn=construct_scalefree_bipartite_heterograph)

## 15. Pipeline C — CTGAN

In [ ]:
meta = SingleTableMetadata()
meta.detect_from_dataframe(train_df)
for col in categorical_columns + gene_columns:
    meta.update_column(column_name=col, sdtype='categorical')

cls_c  = train_df['Grade'].value_counts()
maj_c  = cls_c.idxmax(); min_c = cls_c.idxmin()
n_need = int(cls_c[maj_c] - cls_c[min_c])
print(f"Generating {n_need} synthetic minority samples via CTGAN...")

synth = CTGANSynthesizer(meta, epochs=150, batch_size=50, verbose=False, cuda=True)
synth.fit(train_df)
cond   = Condition(num_rows=n_need, column_values={'Grade': int(min_c)})
syn_s  = synth.sample_from_conditions(conditions=[cond])
train_ctgan_df = pd.concat([train_df, syn_s], ignore_index=True)
print("CTGAN HPO graph class dist:\n", train_ctgan_df['Grade'].value_counts())

# V13: Pipeline C uses scale-free graph builder
train_ct_graph = to_dev(construct_scalefree_bipartite_heterograph(train_ctgan_df))
run_pipeline('CTGAN', train_graph_hpo=train_ct_graph, augment_fn=apply_ctgan,
             graph_fn=construct_scalefree_bipartite_heterograph)

print("\n✓ All 21 model × pipeline combinations complete (all on SF graphs).")

## 4c. ROS Augmentation Function

In [ ]:
from imblearn.over_sampling import RandomOverSampler


def apply_ros(fold_train_df, seed=42):
    """
    Random Over-Sampling: duplicates minority-class rows at random until balanced.

    - Creates NO synthetic samples (unlike SMOTE) -> safe for binary gene columns
    - Preserves true mutation co-occurrence patterns
    - API matches apply_smote() -> plug-in compatible with run_pipeline()
    """
    feat_cols = gene_columns + ["Gender", "Race", "Age_at_diagnosis"]
    ros = RandomOverSampler(sampling_strategy="auto", random_state=seed)
    Xr, yr = ros.fit_resample(fold_train_df[feat_cols], fold_train_df["Grade"])
    df2 = pd.DataFrame(Xr, columns=feat_cols)
    df2["Grade"] = yr
    for c in gene_columns: df2[c] = df2[c].astype(int)
    df2['Race']   = df2['Race'].astype(int).clip(0, 3)
    df2["Gender"] = df2["Gender"].astype(int).clip(0, 1)
    df2["Race"]   = df2["Race"].astype(int)
    print(f"    [ROS] Grade dist after balancing: {dict(pd.Series(yr).value_counts())}")
    return df2


print(f"Original train Grade dist: {dict(train_df['Grade'].value_counts())}")
_ = apply_ros(train_df.copy())
print("apply_ros() defined and verified.")


## Pipeline D — ROS (Random Over-Sampling)

In [ ]:
print("\n" + "=" * 65)
print("PIPELINE D: ROS (Random Over-Sampling)")
print("=" * 65)
print("Strategy  : duplicate minority-class rows (no interpolation)")
print("Advantage : preserves true mutation co-occurrence structure")
print(f"Original  : {dict(train_df['Grade'].value_counts())}")

_ros_hpo_df     = apply_ros(train_df.copy())
# V13: Pipeline D also uses scale-free graph builder
_train_ros_graph = to_dev(construct_scalefree_bipartite_heterograph(_ros_hpo_df))

if "ROS" not in PIPELINES:
    PIPELINES.append("ROS")
run_pipeline("ROS", train_graph_hpo=_train_ros_graph, augment_fn=apply_ros,
             graph_fn=construct_scalefree_bipartite_heterograph)
print("\nROS pipeline complete.")


## 16. 5-Fold CV Summary Table

In [ ]:
cv_all = pd.concat(cv_results, ignore_index=True)

cv_compact = cv_all.groupby(['Model','Pipeline']).apply(
    lambda g: pd.Series({
        'AUC':       f"{g['auc'].mean():.4f} ± {g['auc'].std():.4f}",
        'Recall-1':  f"{g['recall'].mean():.4f} ± {g['recall'].std():.4f}",
        'Recall-0':  f"{g['recall_0'].mean():.4f} ± {g['recall_0'].std():.4f}",
        'F1':        f"{g['f1'].mean():.4f} ± {g['f1'].std():.4f}",
        'Threshold': f"{g['threshold'].mean():.3f} ± {g['threshold'].std():.3f}",
    })
).reset_index()

print("\n5-FOLD CV SUMMARY (mean ± std across 5 folds)")
print("="*90)
print(cv_compact.to_string(index=False))

## 17. Final Results Table (TCGA Test + CGGA)

**AUC fix**: columns built explicitly by name — no brittle positional assignment.

In [ ]:
# ── Build clean results_df (deduplicated, named columns) ────────
# all_results may have duplicate entries if run_pipeline was called
# multiple times; keep the last entry per (Model, Pipeline, Dataset).
_raw = []
for r in all_results:
    _raw.append({
        'Model':     r['Model'],
        'Pipeline':  r['Pipeline'],
        'Dataset':   r['Dataset'],
        'Threshold': round(r['threshold'], 3),
        'AUC':       round(r['auc'],       4),
        'Accuracy':  round(r['accuracy'],  4),
        'Precision': round(r['precision'], 4),
        'Recall_1':  round(r['recall'],    4),
        'Recall_0':  round(r['recall_0'],  4),
        'F1':        round(r['f1'],        4),
    })

results_df = (pd.DataFrame(_raw)
              .drop_duplicates(subset=['Model','Pipeline','Dataset'], keep='last')
              .reset_index(drop=True))

print("\n" + "="*115)
print("FINAL RESULTS — 7 MODELS × 4 PIPELINES × 2 DATASETS")
print("="*115)
print(results_df.to_string(index=False))

# ── Summary table: best per model (max AUC across pipelines) ─────
print("\n" + "="*115)
print("SUMMARY — BEST PIPELINE PER MODEL (by AUC)")
print("="*115)
for ds in ['TCGA Test', 'CGGA']:
    sub = results_df[results_df.Dataset == ds]
    best = (sub.sort_values('AUC', ascending=False)
               .drop_duplicates('Model')
               [['Model','Pipeline','AUC','Recall_1','Recall_0','F1','Threshold']])
    print(f"\n{ds}:")
    print(best.to_string(index=False))

print("\nTop-5 overall by AUC on TCGA Test:")
print(results_df[results_df.Dataset=='TCGA Test']
      .sort_values('AUC', ascending=False).head(5)
      [['Model','Pipeline','AUC','Recall_1','Recall_0','F1']].to_string(index=False))

print("\nTop-5 overall by AUC on CGGA:")
print(results_df[results_df.Dataset=='CGGA']
      .sort_values('AUC', ascending=False).head(5)
      [['Model','Pipeline','AUC','Recall_1','Recall_0','F1']].to_string(index=False))

# ── Export CSVs ───────────────────────────────────────────────────
results_df.to_csv('V15_results_final.csv', index=False)
print("\n✓ Exported: V15_results_final.csv")

# CV summary CSV (mean ± std per model × pipeline)
cv_all = pd.concat(cv_results, ignore_index=True) if cv_results else pd.DataFrame()
if not cv_all.empty:
    cv_summary = (cv_all.groupby(['Model','Pipeline'])
                  [['auc','accuracy','precision','recall','recall_0','f1','threshold']]
                  .agg(['mean','std'])
                  .round(4))
    # Flatten multi-level columns  e.g. ('auc','mean') -> 'auc_mean'
    cv_summary.columns = ['_'.join(c) for c in cv_summary.columns]
    cv_summary = cv_summary.reset_index()
    cv_summary.to_csv('V15_cv_summary.csv', index=False)
    cv_all.to_csv('V15_cv_folds.csv', index=False)
    print("✓ Exported: V15_cv_summary.csv  (mean±std per model×pipeline)")
    print("✓ Exported: V15_cv_folds.csv    (raw per-fold results)")

## 18. AUC Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ds in zip(axes, ['TCGA Test', 'CGGA']):
    # pivot_table with aggfunc='mean' safely handles any duplicate rows
    pivot = (results_df[results_df.Dataset == ds]
             .pivot_table(index='Model', columns='Pipeline',
                          values='AUC', aggfunc='mean')
             .reindex(columns=['No Balancing', 'SMOTE', 'CTGAN', 'ROS']))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu',
                linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0,
                cbar_kws={'label': 'AUC'})
    ax.set_title(f'AUC — {ds}', fontsize=12)
    ax.set_xlabel('Pipeline'); ax.set_ylabel('Model')
plt.suptitle('AUC Heatmap: All Models × Pipelines', fontsize=13)
plt.tight_layout()
plt.savefig('V16_auc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 19. Recall Heatmaps — Grade-1 and Grade-0 (TCGA Test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [
        (axes[0], 'Recall_1', 'Grade-1 Recall (sensitivity)'),
        (axes[1], 'Recall_0', 'Grade-0 Recall (specificity)')]:
    pivot = (results_df[results_df.Dataset == 'TCGA Test']
             .pivot_table(index='Model', columns='Pipeline',
                          values=col, aggfunc='mean')
             .reindex(columns=['No Balancing', 'SMOTE', 'CTGAN', 'ROS']))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn',
                linewidths=0.5, ax=ax, vmin=0.4, vmax=1.0,
                cbar_kws={'label': col})
    ax.set_title(title); ax.set_xlabel('Pipeline'); ax.set_ylabel('Model')
plt.suptitle('Per-Class Recall Heatmap — TCGA Test', fontsize=13)
plt.tight_layout()
plt.savefig('V15_recall_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 21. ROC Curve Grid

In [ ]:
results_full = pd.DataFrame(all_results)
model_names = [n for n, _, _ in MODEL_REGISTRY]
pipe_colors  = {'No Balancing':'steelblue', 'SMOTE':'forestgreen', 'CTGAN':'tomato', 'ROS':'magenta'}


fig, axes = plt.subplots(len(model_names), 2, figsize=(12, 3.5*len(model_names)))
for row, mname in enumerate(model_names):
    for col, ds in enumerate(['TCGA Test', 'CGGA']):
        ax = axes[row, col]
        for pipe in PIPELINES:
            sub = results_full[(results_full.Model==mname) &
                                (results_full.Pipeline==pipe) &
                                (results_full.Dataset==ds)]
            if sub.empty: continue
            r = sub.iloc[0]
            fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
            ax.plot(fpr, tpr, color=pipe_colors[pipe], lw=1.8,
                    label=f"{pipe} (AUC={r['auc']:.3f})")
        ax.plot([0,1],[0,1],'k--', alpha=0.3)
        ax.set_title(f"{mname} — {ds}", fontsize=8)
        ax.set_xlabel('FPR', fontsize=7); ax.set_ylabel('TPR', fontsize=7)
        ax.legend(fontsize=6); ax.grid(alpha=0.3)
plt.suptitle('ROC Curves: All Models × Pipelines', fontsize=13, y=1.005)
plt.tight_layout()
plt.savefig('V15_roc_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 22. Confusion Matrices

In [ ]:
for ds in ['TCGA Test', 'CGGA']:
    fig, axes = plt.subplots(1, len(PIPELINES), figsize=(5*len(PIPELINES), 4))
    cmap = 'Blues' if ds == 'TCGA Test' else 'Oranges'
    for pi, pipe in enumerate(PIPELINES):
        sub  = results_full[(results_full.Dataset==ds) & (results_full.Pipeline==pipe)]
        if sub.empty: continue
        best = sub.loc[sub['auc'].idxmax()]
        th   = best['threshold']
        preds = (best['probs'] >= th).astype(int)
        cm    = confusion_matrix(best['labels'], preds)
        ConfusionMatrixDisplay(cm, display_labels=['Grade 0','Grade 1']).plot(
            ax=axes[pi], cmap=cmap, colorbar=False)
        axes[pi].set_title(f"{pipe}\n{best['Model']} (AUC={best['auc']:.3f}, th={th:.2f})",
                           fontsize=9)
    plt.suptitle(f'Confusion Matrices — {ds}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'V15_cm_{ds.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 23. Graph-Specific XAI

Global permutation importance is replaced by **graph-native attribution** methods that operate directly on the heterogeneous scale-free structure.

| Method | What it explains | Scope |
|--------|-----------------|-------|
| **Edge Occlusion** | Mask each Gene→Patient edge; measure Δpred_prob | Per-patient, edge-level |
| **Integrated Gradients** | Gradient path from zero-baseline along clinical features | Per-patient, node-feature |
| **Population Graph Attribution** | Aggregated occlusion over balanced patient sample, stratified by grade | Dataset-level |
| **Case Study** | GNNExplainer-style 4-panel per patient + cross-dataset overlap | Instance-level |

All methods operate on the **full dual-layer scale-free graph** (Gene↔Patient bipartite + Gene-Gene BA + Patient-Patient BA).


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

CLINICAL_FEAT_NAMES = ['Gender', 'Race_White', 'Race_Black', 'Race_Asian', 'Race_NativeAm', 'Age_norm']


# ============================================================
# XAI Helper 1 -- Edge Occlusion Attribution
# Masks each Gene->Patient edge one at a time; measures delta pred_prob.
# Works for ALL 7 architectures -- no model internals required.
# Respects the full dual-layer SF graph (gene-gene + patient-patient).
# ============================================================
def edge_occlusion_attribution(model, graph, patient_idx):
    """
    Returns: (importances: dict {gene_name: delta_prob}, base_prob: float)
    delta > 0 means removing that edge reduced Grade-1 prob -> gene was helping.
    """
    model.eval()
    cpu_ei = graph[('Gene','mutates','Patient')].edge_index.cpu()

    with torch.no_grad():
        base_prob = F.softmax(model(graph), 1)[patient_idx, 1].item()

    pat_edge_idxs = (cpu_ei[1] == patient_idx).nonzero(as_tuple=True)[0].tolist()
    if not pat_edge_idxs:
        return {}, base_prob

    importances = {}
    orig_g2p = graph[('Gene','mutates','Patient')].edge_index
    has_p2g  = ('Patient','mutated_by','Gene') in graph.edge_index_dict
    if has_p2g:
        orig_p2g = graph[('Patient','mutated_by','Gene')].edge_index
        cpu_p2g  = orig_p2g.cpu()

    for e_idx in pat_edge_idxs:
        gene_idx  = int(cpu_ei[0, e_idx])
        gene_name = gene_columns[gene_idx] if gene_idx < len(gene_columns) else f'G{gene_idx}'

        keep_g2p = torch.ones(cpu_ei.shape[1], dtype=torch.bool)
        keep_g2p[e_idx] = False
        graph[('Gene','mutates','Patient')].edge_index = orig_g2p[:, keep_g2p].to(device)

        if has_p2g:
            keep_p2g = ~((cpu_p2g[0] == patient_idx) & (cpu_p2g[1] == gene_idx))
            graph[('Patient','mutated_by','Gene')].edge_index = orig_p2g[:, keep_p2g].to(device)

        with torch.no_grad():
            try:
                mod_prob = F.softmax(model(graph), 1)[patient_idx, 1].item()
            except Exception:
                mod_prob = base_prob

        graph[('Gene','mutates','Patient')].edge_index = orig_g2p
        if has_p2g:
            graph[('Patient','mutated_by','Gene')].edge_index = orig_p2g

        importances[gene_name] = round(base_prob - mod_prob, 6)

    return importances, base_prob


# ============================================================
# XAI Helper 2 -- Integrated Gradients on Clinical Features
# Patient.x = [Gender, Race, Age_at_diagnosis] (3-dim)
# ============================================================
def integrated_gradients_clinical(model, graph, patient_idx, n_steps=50):
    """Returns ig array [3] aligned with CLINICAL_FEAT_NAMES."""
    model.eval()
    x_orig = graph['Patient'].x.detach()
    x_act  = x_orig[patient_idx]
    label  = int(graph['Patient'].y[patient_idx])
    acc_g  = torch.zeros(x_act.shape[0], device=device)  # 6 features

    for step in range(n_steps):
        alpha = step / max(n_steps - 1, 1)
        x_mod = x_orig.clone()
        x_mod[patient_idx] = alpha * x_act
        x_mod.requires_grad_(True)
        old_x = graph['Patient'].x
        graph['Patient'].x = x_mod
        score = F.softmax(model(graph), 1)[patient_idx, label]
        score.backward()
        graph['Patient'].x = old_x
        if x_mod.grad is not None:
            acc_g += x_mod.grad[patient_idx].detach()

    return (x_act * acc_g / n_steps).cpu().numpy()


# ============================================================
# XAI Helper 3 -- Population-Level Graph Attribution
# Aggregates edge occlusion over balanced patient sample.
# ============================================================
def population_graph_attribution(model, graph, sample_n=60):
    """Returns DataFrame with columns [patient_idx, gene, delta_prob, grade]."""
    model.eval()
    labels = graph['Patient'].y.cpu().numpy()
    rng    = np.random.default_rng(0)
    idx0   = np.where(labels == 0)[0]
    idx1   = np.where(labels == 1)[0]
    half   = min(sample_n // 2, len(idx0), len(idx1))
    sel    = np.concatenate([rng.choice(idx0, half, replace=False),
                              rng.choice(idx1, half, replace=False)])
    records = []
    for pidx in sel:
        imp, _ = edge_occlusion_attribution(model, graph, int(pidx))
        for gname, delta in imp.items():
            records.append({'patient_idx': int(pidx), 'gene': gname,
                            'delta_prob': delta, 'grade': int(labels[pidx])})
    return pd.DataFrame(records)


print("Graph XAI helpers defined: edge_occlusion_attribution,")
print("  integrated_gradients_clinical, population_graph_attribution")


In [ ]:
# ============================================================
# Population-Level Graph Attribution -- run on best model
# ============================================================
_rf = pd.DataFrame([{'Model': r['Model'], 'Pipeline': r['Pipeline'],
                      'Dataset': r['Dataset'], 'auc': r['auc']}
                     for r in all_results])
_best_row    = _rf[_rf.Dataset == 'TCGA Test'].sort_values('auc', ascending=False).iloc[0]
XAI_MODEL_NAME = _best_row['Model']
XAI_PIPELINE   = _best_row['Pipeline']
xai_model      = all_models[(XAI_MODEL_NAME, XAI_PIPELINE)]
xai_threshold  = all_thresholds[(XAI_MODEL_NAME, XAI_PIPELINE)]
print(f"Best model: {XAI_MODEL_NAME} / {XAI_PIPELINE}  (TCGA AUC={_best_row['auc']:.4f})")

print("Running population graph attribution on TCGA test set (n=60 balanced)...")
pop_attr_df = population_graph_attribution(xai_model, test_graph, sample_n=60)
print(f"  {len(pop_attr_df)} attribution records collected")

# Mean delta by (gene, grade)
mean_by_gg = (pop_attr_df.groupby(['gene','grade'])['delta_prob']
              .mean().unstack('grade').fillna(0)
              .reindex(columns=[0,1]).rename(columns={0:'Grade 0', 1:'Grade 1'}))
mean_by_gg = mean_by_gg.sort_values('Grade 1', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Grouped bar -- Gene importance by grade
x = np.arange(len(mean_by_gg)); w = 0.38
axes[0].bar(x - w/2, mean_by_gg['Grade 0'], w, label='Grade 0',
            color='#4C8AC4', edgecolor='white', alpha=0.85)
axes[0].bar(x + w/2, mean_by_gg['Grade 1'], w, label='Grade 1',
            color='#E87722', edgecolor='white', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(mean_by_gg.index, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Mean delta Prob (Edge Occlusion)')
axes[0].set_title(f'Population-Level Gene Edge Importance\n{XAI_MODEL_NAME}/{XAI_PIPELINE}', fontsize=10)
axes[0].axhline(0, color='#333', lw=0.7, ls='--')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)

# Heatmap
sns.heatmap(mean_by_gg.T, annot=True, fmt='.4f', cmap='RdYlGn_r',
            linewidths=0.4, ax=axes[1], cbar_kws={'label': 'Mean delta Prob'})
axes[1].set_title('Gene Edge Importance Heatmap\n(Grade 0 vs Grade 1)', fontsize=10)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle('Graph-Specific XAI: Population Edge Attribution\n'
             '(Dual Scale-Free Graph, n=60 balanced test patients)', fontsize=12)
plt.tight_layout()
plt.savefig('V16_xai_population_attribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: V16_xai_population_attribution.png")

top5 = mean_by_gg['Grade 1'].nlargest(5)
print("\nTop-5 Grade-1 driver genes (mean delta prob):")
for gn, v in top5.items():
    print(f"  {gn:12s}: {v:+.4f}")


## 23b. Feature Importance — Clinical + Gene (Population Level)

Aggregates attribution scores across all TCGA test patients:

- **Gene importance**: mean edge-occlusion delta-prob per gene, split by grade  *(already computed above)*
- **Clinical importance**: mean |Integrated Gradients| per feature (Gender, Race×4, Age) across balanced sample
- **Unified ranking**: all features ranked together by absolute importance


In [ ]:
# ============================================================
# Population-Level Clinical Feature Importance (Integrated Gradients)
# Runs IG on a balanced sample, averages |IG| per feature per grade,
# then combines with gene edge-occlusion into a single unified ranking.
# ============================================================

def population_clinical_importance(model, graph, sample_n=60):
    """Returns DataFrame: columns = CLINICAL_FEAT_NAMES, rows = patients."""
    model.eval()
    labels = graph['Patient'].y.cpu().numpy()
    rng    = np.random.default_rng(42)
    idx0   = np.where(labels == 0)[0]
    idx1   = np.where(labels == 1)[0]
    half   = min(sample_n // 2, len(idx0), len(idx1))
    sel    = np.concatenate([rng.choice(idx0, half, replace=False),
                              rng.choice(idx1, half, replace=False)])
    records = []
    for pidx in sel:
        ig = integrated_gradients_clinical(model, graph, int(pidx), n_steps=30)
        row = {'patient_idx': int(pidx), 'grade': int(labels[pidx])}
        for fname, val in zip(CLINICAL_FEAT_NAMES, ig):
            row[fname] = float(val)
        records.append(row)
    return pd.DataFrame(records)

print("Running population clinical IG (n=60 balanced)...")
clin_df = population_clinical_importance(xai_model, test_graph, sample_n=60)
print(f"  Done. {len(clin_df)} patients processed.")

# ── Mean |IG| per feature per grade ──────────────────────────────
feat_cols  = CLINICAL_FEAT_NAMES
abs_mean   = clin_df.groupby('grade')[feat_cols].apply(lambda d: d.abs().mean())
abs_mean.index = ['Grade 0', 'Grade 1']

# ── Signed mean per grade (for direction) ────────────────────────
sign_mean  = clin_df.groupby('grade')[feat_cols].mean()
sign_mean.index = ['Grade 0', 'Grade 1']

# ── Overall mean |IG| across both grades ─────────────────────────
overall_clin = clin_df[feat_cols].abs().mean().sort_values(ascending=False)

# ── Combine gene + clinical into unified ranking ──────────────────
# Gene: use mean |delta_prob| collapsed across both grades
if 'pop_attr_df' in dir() and len(pop_attr_df):
    gene_imp = pop_attr_df.groupby('gene')['delta_prob'].apply(
        lambda x: x.abs().mean()).sort_values(ascending=False)
else:
    gene_imp = pd.Series(dtype=float)

unified = pd.concat([
    gene_imp.rename('importance').reset_index().rename(columns={'gene':'feature'}),
    pd.DataFrame({'feature': overall_clin.index, 'importance': overall_clin.values})
]).sort_values('importance', ascending=False).reset_index(drop=True)
unified['type'] = unified['feature'].apply(
    lambda f: 'Clinical' if f in CLINICAL_FEAT_NAMES else 'Gene Mutation')

# ── Plot: 3 panels ────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 12))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.38,
                        left=0.05, right=0.98, top=0.91, bottom=0.10)
ax1 = fig.add_subplot(gs[0, :2])   # unified ranked bar (top, wide)
ax2 = fig.add_subplot(gs[0, 2])    # clinical |IG| grouped bar
ax3 = fig.add_subplot(gs[1, :])    # clinical signed heatmap (full width)

# ── Panel 1: Unified ranked feature importance ────────────────────
colors_u = ['#e84393' if t == 'Clinical' else '#3498db' for t in unified['type']]
bars = ax1.barh(unified['feature'][::-1], unified['importance'][::-1],
                color=colors_u[::-1], edgecolor='white', alpha=0.88)
ax1.set_xlabel('Mean |Attribution Score|', fontsize=10)
ax1.set_title(f'Unified Feature Importance — All Genes + Clinical Features\n'
              f'({XAI_MODEL_NAME}/{XAI_PIPELINE}, TCGA Test)', fontsize=10)
ax1.grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color='#3498db', label='Gene Mutation (Edge Occlusion)'),
                    Patch(color='#e84393', label='Clinical (Integrated Gradients)')],
           fontsize=9, loc='lower right')

# ── Panel 2: Clinical |IG| by grade ──────────────────────────────
x = np.arange(len(feat_cols)); w = 0.38
ax2.bar(x - w/2, abs_mean.loc['Grade 0'], w, label='Grade 0',
        color='#4C8AC4', edgecolor='white', alpha=0.85)
ax2.bar(x + w/2, abs_mean.loc['Grade 1'], w, label='Grade 1',
        color='#E87722', edgecolor='white', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(feat_cols, rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('Mean |IG Score|', fontsize=9)
ax2.set_title('Clinical Feature Importance by Grade\n(Mean |Integrated Gradients|)', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# ── Panel 3: Signed clinical IG heatmap (direction matters) ──────
sns.heatmap(sign_mean[feat_cols], annot=True, fmt='.4f',
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax3,
            cbar_kws={'label': 'Mean IG Score (+ = pushes toward Grade 1)'})
ax3.set_title('Clinical Feature Attribution Direction\n'
              '(+ = increases Grade-1 probability, − = decreases it)', fontsize=10)
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=35, ha='right', fontsize=9)
ax3.set_yticklabels(ax3.get_yticklabels(), rotation=0, fontsize=9)

plt.suptitle('Population-Level Feature Importance\n'
             'Gene Mutations (Edge Occlusion) + Clinical Features (Integrated Gradients)',
             fontsize=13, y=0.98)
plt.savefig('V16_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: V16_feature_importance.png")

# ── Text summary ─────────────────────────────────────────────────
print(f"\nTop-5 overall features (combined ranking):")
for _, row in unified.head(5).iterrows():
    print(f"  [{row['type']:16s}] {row['feature']:15s}: {row['importance']:.4f}")

print(f"\nClinical feature ranking (|IG|):")
for fname, val in overall_clin.items():
    g0 = abs_mean.loc['Grade 0', fname]
    g1 = abs_mean.loc['Grade 1', fname]
    print(f"  {fname:18s}: overall={val:.4f}  G0={g0:.4f}  G1={g1:.4f}")


## 24. Case Study — GNNExplainer on High-Confidence Patients

Picks the TCGA Test and CGGA patient predicted correctly with the highest confidence for Grade 1, then runs edge occlusion + integrated gradients and overlays both patients' attributions to find **cross-dataset consistent driver genes**.

In [ ]:
# ============================================================
# Case Study Helpers
# ============================================================

def find_high_confidence_patient(model, graph, threshold, target_grade=1):
    """Returns (patient_idx, pred_prob, true_label) for highest-confidence correct prediction."""
    model.eval()
    with torch.no_grad():
        probs  = F.softmax(model(graph), 1)[:, 1].cpu().numpy()
    labels = graph['Patient'].y.cpu().numpy()
    preds  = (probs >= threshold).astype(int)
    correct = preds == labels
    mask    = correct & (labels == target_grade)
    if not mask.any():
        mask = correct
    candidates = np.where(mask, probs, -1.0) if target_grade == 1 else np.where(mask, probs, 2.0)
    idx = int(np.argmax(candidates)) if target_grade == 1 else int(np.argmin(candidates))
    return idx, float(probs[idx]), int(labels[idx])


def visualize_case_study(patient_idx, base_prob, true_label,
                          edge_importances, ig_attrs, ref_df,
                          dataset='TCGA', threshold=0.5):
    """4-panel: bipartite subgraph | gene bar | clinical IG | patient profile table."""
    pred_label  = int(base_prob >= threshold)
    correct_str = 'Correct' if pred_label == true_label else 'Wrong'

    fig = plt.figure(figsize=(22, 9))
    gs  = fig.add_gridspec(2, 3, hspace=0.50, wspace=0.38,
                            left=0.04, right=0.98, top=0.90, bottom=0.08)
    ax_g   = fig.add_subplot(gs[:, 0])
    ax_bar = fig.add_subplot(gs[0, 1])
    ax_cln = fig.add_subplot(gs[0, 2])
    ax_tbl = fig.add_subplot(gs[1, 1:])

    # Panel A -- bipartite explanation subgraph
    ax_g.axis('off')
    ax_g.set_title(f'Explanation Subgraph\n{dataset} Patient {patient_idx}', fontsize=9, pad=6)
    mutated = list(edge_importances.keys())
    n_g = len(mutated)
    if n_g > 0:
        gene_y = np.linspace(0.06, 0.94, n_g)
        gene_x = np.full(n_g, 0.08)
        pat_x, pat_y = 0.82, 0.50
        vals = np.array(list(edge_importances.values()))
        vmax = max(abs(vals).max(), 1e-6)
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        cmap = cm.RdYlGn_r
        for i, (gname, delta) in enumerate(edge_importances.items()):
            c  = cmap(norm(delta))
            lw = 0.6 + 3.8 * abs(delta) / vmax
            ax_g.annotate("", xy=(pat_x - 0.05, pat_y), xytext=(gene_x[i] + 0.05, gene_y[i]),
                           arrowprops=dict(arrowstyle='-', color=c, lw=lw, alpha=0.82))
        for i, gname in enumerate(mutated):
            c = cmap(norm(edge_importances[gname]))
            ax_g.scatter(gene_x[i], gene_y[i], s=140, color=c, edgecolors='#333', lw=0.7, zorder=4)
            ax_g.text(gene_x[i] - 0.03, gene_y[i], gname, ha='right', va='center',
                       fontsize=7.5, fontfamily='monospace')
        pcol = '#c0392b' if true_label == 1 else '#2980b9'
        ax_g.scatter(pat_x, pat_y, s=420, color=pcol, edgecolors='#111', lw=1.5, zorder=4)
        ax_g.text(pat_x + 0.05, pat_y,
                   f"P{patient_idx}\nGrade {true_label}\nProb={base_prob:.3f}",
                   ha='left', va='center', fontsize=8)
        sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax_g, fraction=0.025, pad=0.01, aspect=30)
        cbar.set_label('delta Prob', fontsize=7); cbar.ax.tick_params(labelsize=6)
        ax_g.set_xlim(-0.05, 1.05); ax_g.set_ylim(-0.02, 1.02)

    # Panel B -- gene edge occlusion bar
    if edge_importances:
        gs_sorted = sorted(edge_importances.items(), key=lambda x: x[1], reverse=True)
        gnames, gvals = zip(*gs_sorted)
        bcols = ['#c0392b' if v > 0 else '#27ae60' for v in gvals]
        ax_bar.barh(gnames, gvals, color=bcols, edgecolor='white', height=0.65)
        ax_bar.axvline(0, color='#333', lw=0.8, ls='--')
        ax_bar.set_xlabel('delta Prob (baseline - masked)', fontsize=8)
        ax_bar.set_title('Gene Edge Importance\n(Edge Occlusion)', fontsize=9)
        ax_bar.tick_params(axis='y', labelsize=7.5); ax_bar.grid(axis='x', alpha=0.3)

    # Panel C -- clinical IG
    if ig_attrs is not None and len(ig_attrs) == len(CLINICAL_FEAT_NAMES):
        ccols = ['#c0392b' if v > 0 else '#27ae60' for v in ig_attrs]
        ax_cln.bar(CLINICAL_FEAT_NAMES, ig_attrs, color=ccols, edgecolor='white', width=0.5)
        ax_cln.axhline(0, color='#333', lw=0.8, ls='--')
        ax_cln.set_title('Clinical Feature Attribution\n(Integrated Gradients)', fontsize=9)
        ax_cln.set_ylabel('IG Score', fontsize=8); ax_cln.grid(axis='y', alpha=0.3)
        for xi, (fn, v) in enumerate(zip(CLINICAL_FEAT_NAMES, ig_attrs)):
            ax_cln.text(xi, v + (0.001 if v >= 0 else -0.001), f'{v:+.4f}',
                         ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)

    # Panel D -- patient profile table
    ax_tbl.axis('off')
    row_data = ref_df.iloc[patient_idx] if patient_idx < len(ref_df) else pd.Series()
    profile  = [
        ['Dataset', dataset], ['Patient Index', str(patient_idx)],
        ['True Grade', f'Grade {true_label}'],
        ['Predicted Grade', f'Grade {pred_label}'],
        ['Prediction', correct_str], ['Confidence (P1)', f'{base_prob:.4f}'],
        ['Age at Diagnosis', str(row_data.get('Age_at_diagnosis', 'N/A'))],
        ['Gender', str(row_data.get('Gender', 'N/A'))],
        ['Race',   str(row_data.get('Race', 'N/A'))],
    ]
    top_genes = sorted(edge_importances.items(), key=lambda x: x[1], reverse=True)[:5]
    for gname, delta in top_genes:
        profile.append([f'delta {gname}', f'{delta:+.4f}'])
    tbl = ax_tbl.table(cellText=profile, colLabels=['Property', 'Value'],
                        cellLoc='left', loc='center', bbox=[0.0, 0.0, 1.0, 1.0])
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0: cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0: cell.set_facecolor('#ecf0f1')
        cell.set_edgecolor('#bdc3c7')

    fig.suptitle(f'{dataset} Case Study -- {correct_str} | {XAI_MODEL_NAME}/{XAI_PIPELINE}',
                  fontsize=12, fontweight='bold')
    fname = f'V16_case_study_{dataset.replace(" ","_")}_P{patient_idx}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.show()
    print(f"  Saved: {fname}")
    return fname


print("Case-study helpers defined: find_high_confidence_patient, visualize_case_study")


In [ ]:
# ============================================================
# Run Case Studies -- TCGA + CGGA + Overlapping Gene Analysis
# ============================================================
print("=" * 65)
print("CASE STUDY -- GNNExplainer-style on High-Confidence Grade-1 Patients")
print(f"Model: {XAI_MODEL_NAME} / {XAI_PIPELINE}  |  Threshold: {xai_threshold:.3f}")
print("=" * 65)

print("\n[1/2] TCGA Test...")
tcga_idx, tcga_prob, tcga_true = find_high_confidence_patient(xai_model, test_graph, xai_threshold)
print(f"  Patient {tcga_idx} | True={tcga_true} | Prob={tcga_prob:.4f}")
tcga_edge_imp, _ = edge_occlusion_attribution(xai_model, test_graph, tcga_idx)
tcga_ig          = integrated_gradients_clinical(xai_model, test_graph, tcga_idx)
visualize_case_study(tcga_idx, tcga_prob, tcga_true, tcga_edge_imp, tcga_ig,
                      test_df, dataset='TCGA Test', threshold=xai_threshold)

print("\n[2/2] CGGA...")
cgga_idx, cgga_prob, cgga_true = find_high_confidence_patient(xai_model, cgga_graph, xai_threshold)
print(f"  Patient {cgga_idx} | True={cgga_true} | Prob={cgga_prob:.4f}")
cgga_edge_imp, _ = edge_occlusion_attribution(xai_model, cgga_graph, cgga_idx)
cgga_ig          = integrated_gradients_clinical(xai_model, cgga_graph, cgga_idx)
visualize_case_study(cgga_idx, cgga_prob, cgga_true, cgga_edge_imp, cgga_ig,
                      cgga_df, dataset='CGGA', threshold=xai_threshold)

# ---- Overlapping Gene Analysis --------------------------------
print("\n" + "=" * 65)
print("OVERLAPPING GENE ANALYSIS -- TCGA vs CGGA Case Study Patients")
print("=" * 65)

all_g     = sorted(set(tcga_edge_imp) | set(cgga_edge_imp))
tcga_vals = np.array([tcga_edge_imp.get(g, 0.0) for g in all_g])
cgga_vals = np.array([cgga_edge_imp.get(g, 0.0) for g in all_g])
sort_idx  = np.argsort(-(tcga_vals + cgga_vals))
all_g     = [all_g[i] for i in sort_idx]
tv        = tcga_vals[sort_idx]; cv = cgga_vals[sort_idx]

THRESH_IMP    = 0.003
is_tcga_imp   = tv > THRESH_IMP
is_cgga_imp   = cv > THRESH_IMP
is_both       = is_tcga_imp & is_cgga_imp
overlap_genes = [g for g, b in zip(all_g, is_both) if b]

x = np.arange(len(all_g)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(20, 5))

# Grouped bars
tcols = ['#c0392b' if b else '#4C8AC4' for b in is_both]
ccols = ['#c0392b' if b else '#E87722' for b in is_both]
axes[0].bar(x - w/2, tv, w, color=tcols, edgecolor='white', alpha=0.88,
             label=f'TCGA P{tcga_idx}')
axes[0].bar(x + w/2, cv, w, color=ccols, edgecolor='white', alpha=0.88,
             label=f'CGGA P{cgga_idx}')
axes[0].set_xticks(x)
axes[0].set_xticklabels(all_g, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('delta Prob (Edge Occlusion)')
axes[0].axhline(THRESH_IMP, color='#aaa', lw=0.8, ls=':', label=f'Threshold ({THRESH_IMP})')
axes[0].axhline(0, color='#333', lw=0.6, ls='--')
axes[0].set_title('Gene Edge Importance: TCGA vs CGGA\n(Red = important in BOTH)', fontsize=10)
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

# Scatter
sc_cols = ['#c0392b' if b else ('#4C8AC4' if t else ('#E87722' if c else '#cccccc'))
           for b, t, c in zip(is_both, is_tcga_imp, is_cgga_imp)]
axes[1].scatter(tv, cv, c=sc_cols, s=90, edgecolors='#333', linewidths=0.5, alpha=0.85, zorder=3)
for i, g in enumerate(all_g):
    if is_both[i] or is_tcga_imp[i] or is_cgga_imp[i]:
        axes[1].annotate(g, (tv[i], cv[i]), textcoords='offset points',
                          xytext=(4, 2), fontsize=7.5)
axes[1].axhline(THRESH_IMP, color='#aaa', lw=0.8, ls=':')
axes[1].axvline(THRESH_IMP, color='#aaa', lw=0.8, ls=':')
axes[1].axhline(0, color='#333', lw=0.6, ls='--')
axes[1].axvline(0, color='#333', lw=0.6, ls='--')
axes[1].set_xlabel(f'TCGA P{tcga_idx} -- delta Prob', fontsize=9)
axes[1].set_ylabel(f'CGGA P{cgga_idx} -- delta Prob', fontsize=9)
axes[1].set_title('Cross-Dataset Gene Attribution Scatter\n'
                   'Top-right quadrant = consistent drivers in both datasets', fontsize=10)

from matplotlib.patches import Patch as MPatch
axes[1].legend(handles=[
    MPatch(color='#c0392b', label='Important in BOTH'),
    MPatch(color='#4C8AC4', label='TCGA only'),
    MPatch(color='#E87722', label='CGGA only'),
    MPatch(color='#cccccc', label='Neither'),
], fontsize=8, loc='lower right')
axes[1].grid(alpha=0.3)

plt.suptitle(f'Cross-Dataset Case Study -- Overlapping Driver Genes\n'
              f'{XAI_MODEL_NAME}/{XAI_PIPELINE} | TCGA P{tcga_idx} vs CGGA P{cgga_idx}',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('V16_case_study_overlapping_genes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: V16_case_study_overlapping_genes.png")

print(f"\nOverlapping driver genes (delta > {THRESH_IMP} in both):")
if overlap_genes:
    for g in overlap_genes:
        print(f"  {g:12s}  TCGA: {tcga_edge_imp.get(g,0):+.4f}   CGGA: {cgga_edge_imp.get(g,0):+.4f}")
else:
    print("  None -- distinct mutation profiles between patients.")

print(f"TCGA-only: {[g for g,t,b in zip(all_g,is_tcga_imp,is_both) if t and not b]}")
print(f"CGGA-only: {[g for g,c,b in zip(all_g,is_cgga_imp,is_both) if c and not b]}")


## 26. Attention Weight Analysis -- HeteroGATv2 & MOGAT

Comprehensive multi-view analysis for both GATv2-based architectures.

| View | What it shows |
|------|--------------|
| **Mean Gene Attention** | Per-gene mean weight with std error bars (sorted by importance) |
| **Grade-Stratified Heatmap** | Mean attention by grade + differential delta bar chart |
| **Per-Head Breakdown** | Each GATv2 head separately; head-to-head correlation matrix |
| **Attention Entropy** | Shannon entropy per gene -- low=selective, high=diffuse |
| **Patient Attention Profile** | Total received attention per patient; violin by grade |
| **HeteroGATv2 Bidirectional** | g2p (Gene->Patient) AND p2g (Patient->Gene) directions |
| **MOGAT Full Fusion Gate** | Genomic vs clinical gate histograms, box plots, gate x attention scatter |
| **Cross-Pipeline Stability** | Pearson correlation + rank plot of gene attention across pipelines |


In [ ]:
# ============================================================
# Shared attention extraction helpers
# ============================================================

def _extract_attn(model, graph):
    """Mean-head Gene->Patient attention. Returns (gene_ids, pat_ids, weights) as numpy arrays."""
    model.eval()
    with torch.no_grad():
        eidx, alpha = model.get_attn_weights(graph)
    return eidx[0].cpu().numpy(), eidx[1].cpu().numpy(), alpha.cpu().numpy()


def _build_attn_matrix(gene_ids, pat_ids, weights, n_genes, n_pat):
    """Sparse edge list -> dense [n_genes, n_pat] attention matrix."""
    mat = np.zeros((n_genes, n_pat))
    for g, p, w in zip(gene_ids, pat_ids, weights):
        if 0 <= g < n_genes and 0 <= p < n_pat:
            mat[g, p] = w
    return mat


def _gene_mean_std(mat):
    """Per-gene mean and std, computed over patients that have at least one edge."""
    means, stds = [], []
    for row in mat:
        nz = row[row > 0]
        means.append(float(nz.mean()) if len(nz) else 0.0)
        stds.append(float(nz.std())   if len(nz) else 0.0)
    return np.array(means), np.array(stds)


def _gene_entropy(mat, eps=1e-9):
    """Shannon entropy (bits) of each gene's attention distribution over patients."""
    out = []
    for row in mat:
        p = row + eps; p /= p.sum()
        out.append(float(-np.sum(p * np.log2(p))))
    return np.array(out)


# ============================================================
# Plot 1 -- Mean Gene Attention Comparison (HeteroGATv2 vs MOGAT)
# ============================================================
def plot_mean_gene_attention_comparison(models_dict, graph, dataset_label='TCGA Test'):
    """Side-by-side mean gene attention bars with std error bars."""
    n = len(models_dict)
    fig, axes = plt.subplots(1, n, figsize=(11 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, (mname, model) in zip(axes, models_dict.items()):
        gids, pids, ws = _extract_attn(model, graph)
        n_p  = graph['Patient'].x.shape[0]
        mat  = _build_attn_matrix(gids, pids, ws, NUM_GENES, n_p)
        means, stds = _gene_mean_std(mat)

        order  = np.argsort(means)[::-1]
        gnames = [gene_columns[i] for i in order]
        gm     = means[order]; gs = stds[order]

        norm   = plt.Normalize(gm.min(), gm.max())
        colors = plt.cm.YlOrRd(norm(gm))
        x      = np.arange(len(gnames))

        ax.bar(x, gm, color=colors, edgecolor='white', zorder=3)
        ax.errorbar(x, gm, yerr=gs, fmt='none',
                    ecolor='#333', elinewidth=0.9, capsize=3, zorder=4)
        ax.set_xticks(x)
        ax.set_xticklabels(gnames, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Mean Attention Weight (+/-std)', fontsize=9)
        ax.set_title(f'{mname}\nMean Gene->Patient Attention', fontsize=10)
        ax.grid(axis='y', alpha=0.3, zorder=0)
        for xi, (m, s) in enumerate(zip(gm, gs)):
            if m > 0.001:
                ax.text(xi, m + s + 0.0005, f'{m:.3f}',
                        ha='center', va='bottom', fontsize=6, rotation=90)

        sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.01, label='Attn weight')

    plt.suptitle(f'Mean Gene->Patient Attention Weights -- {dataset_label}',
                  fontsize=12, y=1.02)
    plt.tight_layout()
    fname = f'V15_attn_mean_cmp_{dataset_label.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


# ============================================================
# Plot 2 -- Grade-Stratified Attention Heatmap + Delta Bar
# ============================================================
def plot_grade_stratified_attention(model, graph, ref_df, title):
    """Heatmap of mean attention by (gene, grade) + differential delta bar."""
    model.eval()
    gids, pids, ws = _extract_attn(model, graph)
    labels = graph['Patient'].y.cpu().numpy()
    n_p    = graph['Patient'].x.shape[0]
    mat    = _build_attn_matrix(gids, pids, ws, NUM_GENES, n_p)

    g0_mean = mat[:, labels == 0].mean(axis=1)
    g1_mean = mat[:, labels == 1].mean(axis=1)
    delta   = g1_mean - g0_mean

    df_h = pd.DataFrame({'Grade 0': g0_mean, 'Grade 1': g1_mean,
                          'Delta': delta}, index=gene_columns)
    df_h = df_h.sort_values('Grade 1', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                              gridspec_kw={'width_ratios': [2, 1]})

    sns.heatmap(df_h[['Grade 0', 'Grade 1']], annot=True, fmt='.4f',
                cmap='YlOrRd', linewidths=0.4, ax=axes[0],
                cbar_kws={'label': 'Mean Attention'})
    axes[0].set_title(f'Gene Attention by Grade\n{title}', fontsize=10)
    axes[0].tick_params(axis='y', labelsize=8)

    dcols = ['#c0392b' if v > 0 else '#2980b9' for v in df_h['Delta']]
    axes[1].barh(df_h.index, df_h['Delta'], color=dcols, edgecolor='white', height=0.7)
    axes[1].axvline(0, color='#333', lw=0.8, ls='--')
    axes[1].set_xlabel('Delta Attention (G1 - G0)', fontsize=8)
    axes[1].set_title('Differential Attention\nGrade 1 vs Grade 0', fontsize=10)
    axes[1].tick_params(axis='y', labelsize=8)
    axes[1].grid(axis='x', alpha=0.3)
    for bar_p, val in zip(axes[1].patches, df_h['Delta']):
        xv = bar_p.get_width()
        axes[1].text(xv + (0.0002 if xv >= 0 else -0.0002),
                     bar_p.get_y() + bar_p.get_height() / 2,
                     f'{val:+.4f}', va='center',
                     ha='left' if xv >= 0 else 'right', fontsize=6.5)

    plt.suptitle(f'Grade-Stratified Attention -- {title}', fontsize=12)
    plt.tight_layout()
    safe  = title.replace('/', '_').replace(' ', '_')
    fname = f'V15_attn_grade_{safe}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')
    return df_h


# ============================================================
# Plot 3 -- Per-Head Attention Breakdown + Head Correlation
# ============================================================
def plot_per_head_attention(model, graph, title):
    """One sub-plot per GATv2 attention head; head-to-head correlation matrix."""
    model.eval()
    with torch.no_grad():
        eidx, alpha_all = model.get_all_heads_attn(graph)   # [E, H]
    gene_ids = eidx[0].cpu().numpy()
    pat_ids  = eidx[1].cpu().numpy()
    alpha_np = alpha_all.cpu().numpy()
    n_heads  = alpha_np.shape[1]
    n_p      = graph['Patient'].x.shape[0]

    head_gene_means = np.zeros((n_heads, NUM_GENES))
    for h in range(n_heads):
        mat_h = _build_attn_matrix(gene_ids, pat_ids,
                                    alpha_np[:, h], NUM_GENES, n_p)
        head_gene_means[h] = mat_h.mean(axis=1)

    sort_order = np.argsort(head_gene_means.mean(axis=0))[::-1]
    g_labels   = [gene_columns[i] for i in sort_order]
    data_plot  = head_gene_means[:, sort_order]

    fig, axes = plt.subplots(n_heads, 1,
                              figsize=(16, 2.8 * n_heads), sharex=True)
    if n_heads == 1:
        axes = [axes]

    for h, ax in enumerate(axes):
        norm   = plt.Normalize(data_plot[h].min(), data_plot[h].max())
        colors = [plt.cm.YlOrRd(norm(v)) for v in data_plot[h]]
        ax.bar(g_labels, data_plot[h], color=colors, edgecolor='white')
        ax.set_ylabel(f'Head {h+1}\nAttn', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        top3 = np.argsort(data_plot[h])[::-1][:3]
        for ti in top3:
            ax.text(ti, data_plot[h, ti] + 0.0003,
                    f'{data_plot[h, ti]:.4f}', ha='center', va='bottom',
                    fontsize=7, color='#c0392b', fontweight='bold')

    axes[-1].set_xticklabels(g_labels, rotation=45, ha='right', fontsize=8)
    axes[-1].set_xlabel('Gene (sorted by mean across heads)', fontsize=9)
    fig.suptitle(f'Per-Head Gene->Patient Attention -- {title}',
                  fontsize=12, y=1.01)
    fig.tight_layout()
    safe  = title.replace('/', '_').replace(' ', '_')
    fig.savefig(f'V15_attn_perhead_{safe}.png', dpi=150, bbox_inches='tight')

    corr = np.corrcoef(head_gene_means)
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                linewidths=0.4, ax=ax2,
                xticklabels=[f'H{h+1}' for h in range(n_heads)],
                yticklabels=[f'H{h+1}' for h in range(n_heads)])
    ax2.set_title(f'Head Correlation Matrix\n{title}', fontsize=9)
    fig2.tight_layout()
    fig2.savefig(f'V15_attn_headcorr_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: V15_attn_perhead_{safe}.png  V15_attn_headcorr_{safe}.png')


# ============================================================
# Plot 4 -- Attention Entropy + Patient Profile
# ============================================================
def plot_entropy_and_patient_profile(model, graph, ref_df, title):
    """
    Left  : Shannon entropy per gene (how focused vs diffuse each gene's attention is).
    Centre: Entropy vs mean attention scatter.
    Right : Violin of total incoming attention per patient, split by grade.
    """
    model.eval()
    gids, pids, ws = _extract_attn(model, graph)
    labels    = graph['Patient'].y.cpu().numpy()
    n_p       = graph['Patient'].x.shape[0]
    mat       = _build_attn_matrix(gids, pids, ws, NUM_GENES, n_p)
    entropies = _gene_entropy(mat)
    means, _  = _gene_mean_std(mat)
    pat_total = mat.sum(axis=0)

    ent_order = np.argsort(entropies)
    fig, axes = plt.subplots(1, 3, figsize=(21, 5))

    # Entropy bar
    ecols = plt.cm.RdYlGn_r(entropies[ent_order] / (entropies.max() + 1e-9))
    axes[0].barh([gene_columns[i] for i in ent_order],
                  entropies[ent_order], color=ecols, edgecolor='white', height=0.75)
    axes[0].set_xlabel('Shannon Entropy (bits)', fontsize=9)
    axes[0].set_title(f'Gene Attention Entropy\n{title}\nLow=selective | High=diffuse', fontsize=9)
    axes[0].grid(axis='x', alpha=0.3)
    for xi, ent in enumerate(entropies[ent_order]):
        axes[0].text(ent + 0.05, xi, f'{ent:.2f}', va='center', fontsize=7)

    # Scatter
    sc_c = plt.cm.YlOrRd(means / (means.max() + 1e-9))
    axes[1].scatter(entropies, means, c=sc_c, s=70,
                     edgecolors='#333', linewidths=0.5, zorder=3)
    for i, g in enumerate(gene_columns):
        axes[1].annotate(g, (entropies[i], means[i]),
                          textcoords='offset points', xytext=(3, 2), fontsize=7)
    axes[1].set_xlabel('Attention Entropy (bits)', fontsize=9)
    axes[1].set_ylabel('Mean Attention Weight', fontsize=9)
    axes[1].set_title(f'Entropy vs Mean Attention\n{title}', fontsize=10)
    axes[1].grid(alpha=0.3, zorder=0)

    # Violin
    g0_data = pat_total[labels == 0]
    g1_data = pat_total[labels == 1]
    vp = axes[2].violinplot([g0_data, g1_data], positions=[0, 1],
                             showmedians=True, showextrema=True)
    for body, col in zip(vp['bodies'], ['#4C8AC4', '#E87722']):
        body.set_facecolor(col); body.set_alpha(0.7)
    for part in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
        if part in vp:
            vp[part].set_color('#333')
    rng = np.random.default_rng(0)
    axes[2].scatter(rng.uniform(-0.04, 0.04, len(g0_data)), g0_data,
                     alpha=0.35, s=12, color='#4C8AC4')
    axes[2].scatter(1 + rng.uniform(-0.04, 0.04, len(g1_data)), g1_data,
                     alpha=0.35, s=12, color='#E87722')
    axes[2].set_xticks([0, 1])
    axes[2].set_xticklabels(['Grade 0', 'Grade 1'], fontsize=9)
    axes[2].set_ylabel('Total Incoming Attention', fontsize=9)
    axes[2].set_title(f'Patient Attention Profile by Grade\n{title}', fontsize=10)
    axes[2].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Attention Entropy & Patient Profile -- {title}', fontsize=12)
    plt.tight_layout()
    safe  = title.replace('/', '_').replace(' ', '_')
    fname = f'V15_attn_entropy_{safe}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


# ============================================================
# Plot 5 -- HeteroGATv2 Bidirectional (g2p + p2g)
# ============================================================
def plot_heterogatv2_bidirectional(model, graph, title):
    """
    Bar chart for g2p attention, bar chart for p2g attention,
    and a scatter showing the correlation between both directions per gene.
    """
    model.eval()
    assert isinstance(model, HeteroGATv2), 'Only for HeteroGATv2'

    gids, pids, g2p_ws = _extract_attn(model, graph)
    n_p = graph['Patient'].x.shape[0]

    with torch.no_grad():
        eidx_r, alpha_r = model.get_p2g_attn_weights(graph)
    p2g_pids = eidx_r[0].cpu().numpy()   # source: patient
    p2g_gids = eidx_r[1].cpu().numpy()   # target: gene
    p2g_ws   = alpha_r.cpu().numpy()

    g2p_mat  = _build_attn_matrix(gids,     pids,     g2p_ws, NUM_GENES, n_p)
    p2g_mat  = _build_attn_matrix(p2g_gids, p2g_pids, p2g_ws, NUM_GENES, n_p)
    g2p_mean = g2p_mat.mean(axis=1)
    p2g_mean = p2g_mat.mean(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(21, 5))

    srt1  = np.argsort(g2p_mean)[::-1]
    norm1 = plt.Normalize(g2p_mean.min(), g2p_mean.max())
    axes[0].bar([gene_columns[i] for i in srt1], g2p_mean[srt1],
                 color=plt.cm.YlOrRd(norm1(g2p_mean[srt1])), edgecolor='white')
    axes[0].set_xticklabels([gene_columns[i] for i in srt1],
                              rotation=45, ha='right', fontsize=8)
    axes[0].set_title(f'Gene->Patient Attention (g2p)\n{title}', fontsize=10)
    axes[0].set_ylabel('Mean Attention Weight')
    axes[0].grid(axis='y', alpha=0.3)

    srt2  = np.argsort(p2g_mean)[::-1]
    norm2 = plt.Normalize(p2g_mean.min(), p2g_mean.max())
    axes[1].bar([gene_columns[i] for i in srt2], p2g_mean[srt2],
                 color=plt.cm.Blues(0.4 + 0.5 * norm2(p2g_mean[srt2])),
                 edgecolor='white')
    axes[1].set_xticklabels([gene_columns[i] for i in srt2],
                              rotation=45, ha='right', fontsize=8)
    axes[1].set_title(f'Patient->Gene Attention (p2g)\n{title}', fontsize=10)
    axes[1].set_ylabel('Mean Attention Weight')
    axes[1].grid(axis='y', alpha=0.3)

    r = float(np.corrcoef(g2p_mean, p2g_mean)[0, 1])
    axes[2].scatter(g2p_mean, p2g_mean, s=70, color='#8e44ad',
                     edgecolors='#333', linewidths=0.5, zorder=3)
    for i, g in enumerate(gene_columns):
        axes[2].annotate(g, (g2p_mean[i], p2g_mean[i]),
                          textcoords='offset points', xytext=(3, 2), fontsize=7)
    axes[2].set_xlabel('g2p Mean Attention', fontsize=9)
    axes[2].set_ylabel('p2g Mean Attention', fontsize=9)
    axes[2].set_title(f'Bidirectional Correlation (r={r:.3f})\n{title}', fontsize=10)
    axes[2].grid(alpha=0.3, zorder=0)

    plt.suptitle(f'HeteroGATv2 Bidirectional Attention -- {title}', fontsize=12)
    plt.tight_layout()
    safe  = title.replace('/', '_').replace(' ', '_')
    fname = f'V15_attn_bidir_{safe}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


# ============================================================
# Plot 6 -- MOGAT Full Fusion Gate (enhanced)
# ============================================================
def plot_mogat_full_gate(model, graph, ref_df, title):
    """
    Row 0: genomic gate histogram | clinical gate histogram | gate scatter
    Row 1: genomic gate box by grade | clinical gate box by grade | gate x attention
    """
    model.eval()
    assert isinstance(model, MOGAT), 'Only for MOGAT'
    labels = graph['Patient'].y.cpu().numpy()

    with torch.no_grad():
        gate_w = model.get_fusion_gate(graph).cpu().numpy()   # [N, 2]

    gids, pids, ws = _extract_attn(model, graph)
    n_p       = graph['Patient'].x.shape[0]
    mat       = _build_attn_matrix(gids, pids, ws, NUM_GENES, n_p)
    pat_max_a = mat.max(axis=0)

    colors = ['#4C8AC4', '#E87722']
    grades = ['Grade 0', 'Grade 1']
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))

    # Row 0 -- histograms
    for col_i, (pathway, pid) in enumerate([('Genomic', 0), ('Clinical', 1)]):
        ax = axes[0, col_i]
        for grade, color, lbl in zip([0, 1], colors, grades):
            data = gate_w[labels == grade, pid]
            ax.hist(data, bins=25, alpha=0.55, color=color,
                    label=lbl, density=True, edgecolor='white')
            ax.axvline(data.mean(), color=color, lw=1.5, ls='--', alpha=0.85)
        ax.set_xlabel(f'{pathway} gate weight', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'{pathway} Pathway Gate\nDistribution by Grade', fontsize=10)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax_sc = axes[0, 2]
    for grade, color, lbl in zip([0, 1], colors, grades):
        mask = labels == grade
        ax_sc.scatter(gate_w[mask, 0], gate_w[mask, 1], c=color,
                       label=lbl, alpha=0.55, s=22, edgecolors='none')
    ax_sc.plot([0, 1], [1, 0], 'k--', alpha=0.3, lw=1)
    ax_sc.set_xlabel('Genomic gate', fontsize=9)
    ax_sc.set_ylabel('Clinical gate', fontsize=9)
    ax_sc.set_title('Pathway Gate per Patient\n(gate0 + gate1 = 1)', fontsize=10)
    ax_sc.legend(fontsize=8); ax_sc.grid(alpha=0.3)

    # Row 1 -- box plots + gate x attention
    for col_i, (pathway, pid) in enumerate([('Genomic', 0), ('Clinical', 1)]):
        ax = axes[1, col_i]
        bp = ax.boxplot([gate_w[labels == g, pid] for g in [0, 1]],
                         patch_artist=True, notch=True,
                         medianprops=dict(color='#111', lw=1.5))
        for patch, col in zip(bp['boxes'], colors):
            patch.set_facecolor(col); patch.set_alpha(0.65)
        ax.set_xticklabels(grades, fontsize=9)
        ax.set_ylabel(f'{pathway} Gate Weight', fontsize=9)
        ax.set_title(f'{pathway} Gate by Grade\nNotched box = median CI', fontsize=10)
        ax.grid(axis='y', alpha=0.3)

    ax_j = axes[1, 2]
    for grade, color, lbl in zip([0, 1], colors, grades):
        mask = labels == grade
        ax_j.scatter(gate_w[mask, 0], pat_max_a[mask], c=color,
                      label=lbl, alpha=0.55, s=22, edgecolors='none')
    r = float(np.corrcoef(gate_w[:, 0], pat_max_a)[0, 1])
    ax_j.set_xlabel('Genomic Gate Weight', fontsize=9)
    ax_j.set_ylabel('Max Gene->Patient Attention', fontsize=9)
    ax_j.set_title(f'Gate vs Attention (r={r:.3f})\n'
                    'Do high-genomic-gate patients receive more attention?', fontsize=9)
    ax_j.legend(fontsize=8); ax_j.grid(alpha=0.3)

    plt.suptitle(f'MOGAT Full Fusion Gate Analysis -- {title}',
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe  = title.replace('/', '_').replace(' ', '_')
    fname = f'V15_mogat_gate_full_{safe}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


# ============================================================
# Plot 7 -- Cross-Pipeline Attention Stability
# ============================================================
def plot_cross_pipeline_stability(mname, graph):
    """
    Left : Pearson correlation heatmap of mean gene attention across pipelines.
    Right: Gene attention rank per pipeline -- overlapping lines = stable ordering.
    """
    pipe_means = {}
    for pipe in PIPELINES:
        m = all_models.get((mname, pipe))
        if m is None or not hasattr(m, 'get_attn_weights'):
            continue
        m.eval()
        gids, pids, ws = _extract_attn(m, graph)
        n_p = graph['Patient'].x.shape[0]
        mat = _build_attn_matrix(gids, pids, ws, NUM_GENES, n_p)
        pipe_means[pipe] = mat.mean(axis=1)

    if len(pipe_means) < 2:
        print(f'  {mname}: fewer than 2 pipelines available -- skipping')
        return

    df_pm = pd.DataFrame(pipe_means, index=gene_columns)
    corr  = df_pm.corr()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                linewidths=0.4, ax=axes[0], vmin=-1, vmax=1,
                cbar_kws={'label': 'Pearson r'})
    axes[0].set_title(f'{mname} -- Cross-Pipeline Attention Correlation\n'
                       'r=1 means same gene-attention pattern across augmentations', fontsize=9)

    df_rank = df_pm.rank(ascending=False)
    pipe_cols = ['#2980b9', '#27ae60', '#e74c3c', '#9b59b6', '#e67e22']
    for (pipe, series), col in zip(df_rank.items(), pipe_cols):
        axes[1].plot(series.values, marker='o', ms=4, lw=1.2,
                     label=pipe, color=col, alpha=0.8)
    axes[1].set_xticks(range(NUM_GENES))
    axes[1].set_xticklabels(gene_columns, rotation=45, ha='right', fontsize=7.5)
    axes[1].set_ylabel('Attention Rank (1=highest)', fontsize=9)
    axes[1].set_title(f'{mname} -- Gene Attention Rank per Pipeline\n'
                       'Overlapping lines = stable ranking across augmentations', fontsize=9)
    axes[1].invert_yaxis(); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.suptitle(f'Cross-Pipeline Attention Stability -- {mname}', fontsize=12)
    plt.tight_layout()
    fname = f'V15_attn_stability_{mname}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname}')


print('Attention functions defined: _extract_attn, _build_attn_matrix, _gene_mean_std,')
print('  _gene_entropy, plot_mean_gene_attention_comparison,')
print('  plot_grade_stratified_attention, plot_per_head_attention,')
print('  plot_entropy_and_patient_profile, plot_heterogatv2_bidirectional,')
print('  plot_mogat_full_gate, plot_cross_pipeline_stability')


In [ ]:
# ============================================================
# Run Full Attention Suite
# ============================================================
import warnings; warnings.filterwarnings('ignore')

# Best pipeline per model by TCGA Test AUC
_tmp = pd.DataFrame([{'Model': r['Model'], 'Pipeline': r['Pipeline'],
                       'auc': r['auc'], 'Dataset': r['Dataset']}
                      for r in all_results])
best_pipe_per_model = (
    _tmp[_tmp.Dataset == 'TCGA Test']
    .sort_values('auc', ascending=False)
    .drop_duplicates('Model')
    .set_index('Model')['Pipeline']
    .to_dict()
)
print('Best pipeline per model (TCGA Test AUC):')
for mn, bp in best_pipe_per_model.items():
    print(f'  {mn:14s} -> {bp}')

hgat_pipe   = best_pipe_per_model.get('HeteroGATv2', '')
mogat_pipe  = best_pipe_per_model.get('MOGAT', '')
hgat_model  = all_models.get(('HeteroGATv2', hgat_pipe))
mogat_model = all_models.get(('MOGAT', mogat_pipe))
assert hgat_model  is not None, 'HeteroGATv2 not found in all_models'
assert mogat_model is not None, 'MOGAT not found in all_models'
print(f'\nHeteroGATv2 best: {hgat_pipe}  |  MOGAT best: {mogat_pipe}')

# ------------------------------------------------------------
# A. Mean Gene Attention -- side-by-side both models
# ------------------------------------------------------------
print('\n' + '='*60 + '\nA. Mean Gene Attention Comparison\n' + '='*60)
plot_mean_gene_attention_comparison(
    {'HeteroGATv2': hgat_model, 'MOGAT': mogat_model}, test_graph, 'TCGA Test')
plot_mean_gene_attention_comparison(
    {'HeteroGATv2': hgat_model, 'MOGAT': mogat_model}, cgga_graph, 'CGGA')

# ------------------------------------------------------------
# B. Grade-Stratified Attention
# ------------------------------------------------------------
print('\n' + '='*60 + '\nB. Grade-Stratified Attention\n' + '='*60)
for mname, model, pipe in [('HeteroGATv2', hgat_model, hgat_pipe),
                             ('MOGAT', mogat_model, mogat_pipe)]:
    for g, d, ds in [(test_graph, test_df, 'TCGA'), (cgga_graph, cgga_df, 'CGGA')]:
        print(f'  {mname} / {ds}')
        plot_grade_stratified_attention(model, g, d, f'{mname}/{pipe}/{ds}')

# ------------------------------------------------------------
# C. Per-Head Attention Breakdown (TCGA)
# ------------------------------------------------------------
print('\n' + '='*60 + '\nC. Per-Head Attention (TCGA)\n' + '='*60)
for mname, model, pipe in [('HeteroGATv2', hgat_model, hgat_pipe),
                             ('MOGAT', mogat_model, mogat_pipe)]:
    print(f'  {mname} / {pipe}')
    plot_per_head_attention(model, test_graph, f'{mname}/{pipe}/TCGA')

# ------------------------------------------------------------
# D. Entropy + Patient Profile
# ------------------------------------------------------------
print('\n' + '='*60 + '\nD. Entropy + Patient Profile\n' + '='*60)
for mname, model, pipe in [('HeteroGATv2', hgat_model, hgat_pipe),
                             ('MOGAT', mogat_model, mogat_pipe)]:
    for g, d, ds in [(test_graph, test_df, 'TCGA'), (cgga_graph, cgga_df, 'CGGA')]:
        print(f'  {mname} / {ds}')
        plot_entropy_and_patient_profile(model, g, d, f'{mname}/{pipe}/{ds}')

# ------------------------------------------------------------
# E. HeteroGATv2 Bidirectional (g2p + p2g)
# ------------------------------------------------------------
print('\n' + '='*60 + '\nE. HeteroGATv2 Bidirectional\n' + '='*60)
for g, ds in [(test_graph, 'TCGA'), (cgga_graph, 'CGGA')]:
    plot_heterogatv2_bidirectional(hgat_model, g,
                                    f'HeteroGATv2/{hgat_pipe}/{ds}')

# ------------------------------------------------------------
# F. MOGAT Full Fusion Gate
# ------------------------------------------------------------
print('\n' + '='*60 + '\nF. MOGAT Full Fusion Gate\n' + '='*60)
for g, d, ds in [(test_graph, test_df, 'TCGA'), (cgga_graph, cgga_df, 'CGGA')]:
    plot_mogat_full_gate(mogat_model, g, d, f'MOGAT/{mogat_pipe}/{ds}')

# ------------------------------------------------------------
# G. Cross-Pipeline Stability
# ------------------------------------------------------------
print('\n' + '='*60 + '\nG. Cross-Pipeline Stability\n' + '='*60)
for mname in ['HeteroGATv2', 'MOGAT']:
    plot_cross_pipeline_stability(mname, test_graph)

print('\nFull attention suite complete.')
print('PNG outputs: V15_attn_mean_cmp_*, V15_attn_grade_*,')
print('             V15_attn_perhead_*, V15_attn_headcorr_*,')
print('             V15_attn_entropy_*, V15_attn_bidir_*,')
print('             V15_mogat_gate_full_*, V15_attn_stability_*')


## 27. Classification Reports — Best Model Overall

In [ ]:
for ds in ['TCGA Test','CGGA']:
    sub  = results_full[results_full.Dataset==ds]
    best = sub.loc[sub['auc'].idxmax()]
    th   = best['threshold']
    preds  = (best['probs'] >= th).astype(int)
    labels = best['labels']
    print(f"{'='*65}")
    print(f"Best on {ds}: {best['Model']} / {best['Pipeline']}")
    print(f"AUC={best['auc']:.4f}  Threshold={th:.3f}")
    print(classification_report(labels, preds, target_names=['Grade 0','Grade 1']))

## 28. Save Results

In [ ]:
# import os
# os.makedirs('saved_models_v15', exist_ok=True)
# for (mname, pipe), model in all_models.items():
#     fn = f"saved_models_v15/{mname}_{pipe.replace(' ','_')}.pth"
#     torch.save(model.state_dict(), fn)

pd.DataFrame([{'Model': mn, 'Pipeline': pp, 'Threshold': th}
               for (mn, pp), th in all_thresholds.items()]).to_csv('V16_thresholds.csv', index=False)
if 'imp_df' in dir() and not imp_df.empty:
    imp_df.to_csv('V16_feature_importance.csv', index=False)
    print("✓ Exported: V16_feature_importance.csv")

print("✓ Saved model weights to saved_models_v15/")
print("\nFinal AUC Pivot:")
print(results_df.pivot_table(index='Model', columns=['Pipeline','Dataset'],
                              values='AUC', aggfunc='mean').round(4).to_string())